# 📝 과제 01: 온라인 쇼핑 상담 Q&A 만들기

## 만들 결과

교안 01에서 노동법 조문으로 만든 과정을 전자상거래법 조문에 적용해, 온라인 쇼핑 상담 챗봇의 파인튜닝 데이터를 만듭니다. 원문 정리와 생성 대상 고르기, Q&A 생성 체인, 5개 문단 시험과 예상 비용, 전체 생성, 근거 검수, 질문 늘리기, 새 질문에 답 만들기, 범위 밖 거절 행, 저장을 교안 01과 같은 순서로 진행하고, 마지막에 단계별 실제 비용을 계산해 조문으로 Q&A 생성의 예상 비용과 비교합니다.

<img src="images/40_gwaje01_pipeline.png" width="1050" alt="제목은 '과제 01: 온라인 쇼핑 상담 Q&A 만들기'입니다. 번호 배지 1~7의 카드 일곱 장이 왼쪽에서 오른쪽으로 화살표로 이어집니다. 1 '전자상거래법 조문'은 파랑 문서입니다. 2 '원문 정리 · 문단 고르기'는 줄마다 초록 체크나 빨강 엑스가 붙은 문서와 그 아래 빨간 취소선을 그은 회색 칩입니다. 3 'Q&A 생성 체인'은 문서 하나에서 화살표로 나오는 주황 카드 세 장입니다. 4 '근거 검수'는 초록 체크와 빨강 엑스가 붙은 카드 두 장입니다. 5 '질문 늘리기 · 새 질문에 답 만들기'는 퍼지는 카드와 남색 상자를 지나 나오는 초록 카드입니다. 6 '범위 밖 거절'은 물음표 말풍선이 정지 표시가 붙은 답 말풍선으로 이어지는 그림입니다. 7 'commerce_qa.jsonl'은 보라 파일입니다.">

- 전자상거래법 39문단의 원문을 정리하고(정리로 바뀐 문단 15개), Q&A를 만들 생성 대상 문단 37개를 고릅니다.
- 조문 하나에서 질문 유형이 붙은 Q&A 1~3개를 받는 체인을 만들고, 주제 태그가 다른 5개 문단으로 먼저 시험해 전체 토큰 수와 생성 대상 전체 문단의 예상 비용을 어림합니다.
- 생성 대상 문단 전체의 Q&A 행을 만들고 주제 태그별 행 수, 질문 유형별 행 수, 문단당 Q&A 수를 확인합니다.
- 답의 숫자 대조와 금지 표현 검사로 근거가 맞지 않는 행을 거릅니다.
- 질문 늘리기로 행이 적은 주제에 이용자나 판매자 말투의 질문을 더 많이 늘리고, 하이브리드 검색(재정렬은 끔)으로 찾은 조문을 보고 생성용 LLM이 새 질문에 근거 조문과 답을 붙입니다.
- 이 챗봇이 다루는 전자상거래법으로 답할 수 없는 질문에 고정 거절 답을 붙인 범위 밖 행을 만듭니다.
- 검수를 통과한 행과 범위 밖 행을 `output/commerce_qa.jsonl`로 저장합니다.
- API를 부른 단계마다 모은 토큰 사용량으로 단계별 실제 비용을 계산하고, 조문으로 Q&A 생성은 예상 비용과 비교합니다.

## 준비: 데이터와 제공 코드

- **자료:** `data/과제/commerce_law_articles.json`
  - 「전자상거래 등에서의 소비자보호에 관한 법률」(전자상거래법)에서 청약확인·공급·청약철회·환급·통신판매중개 조문을 항 단위로 나눈 문단에, 생성 대상 고르기를 연습하도록 다른 장의 문단 두 개를 더한 39문단입니다.
  - 국가법령정보센터([law.go.kr](https://www.law.go.kr)) 원문이고, 저작권법 제7조에 따라 보호받지 않는 저작물입니다.
  - 시행일자는 `data/과제/commerce_law_source.json`에 있습니다.
- **키:** 교안 01과 같은 `doc_id`(예: "전자상거래법 제17조 제1항"), `law`, `source`(주제 태그), `text`(원문)입니다. 주제 태그는 거래조건 표시·청약확인, 공급·배송, 청약철회, 환급·반환 비용, 통신판매중개 다섯 가지입니다.
- **제공 코드:** 경로와 생성용 LLM `llm`을 만드는 준비 셀, 교안 01의 `clean_text`, `DELEGATION_ENDINGS`(전자상거래법이 쓰는 "총리령으로 정한다."를 더함), `is_qa_target`, `numbers_match`, `jaccard`, `sum_total_tokens`, `BANNED_PHRASES`, `has_banned_phrase`, 가격 상수 `INPUT_PRICE`, `CACHED_INPUT_PRICE`, `OUTPUT_PRICE`와 비용을 계산하는 `usage_cost`
  - **`usage_cost(usage_metadata)`:** 교안 01과 같은 함수입니다. `usage_metadata` 하나의 입력 토큰(캐시 입력은 따로), 출력 토큰에 토큰 100만 개당 가격을 곱한 비용(달러)을 돌려줍니다.
- **검색기 파일:** 7번에서 교안 01과 같이 단원 폴더의 `law_retriever.py`에서 `build_retriever`를 불러 씁니다(import 줄은 셀에 적어 두었습니다).
- **작성:** `[작성]` 셀의 `...`를 채우고 바로 아래 `[자가채점]` 셀을 실행하세요. 뒤 문항이 앞 문항의 변수를 쓰므로 1번부터 차례로 풉니다.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"  # 이 단원이 만든 데이터셋 파일을 저장하는 폴더
output_dir.mkdir(exist_ok=True)

# API 키 읽기: 단원 폴더 .env를 먼저 읽고, 없는 키만 상위 실습자료 .env에서 채웁니다
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")

# 데이터를 만드는 생성용 LLM. .env에 OPENAI_CHAT_MODEL이 없으면 기본 모델을 씁니다
# use_responses_api=True: 앞 단원들과 같이 OpenAI Responses API로 호출합니다
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
llm = ChatOpenAI(model=model_name, use_responses_api=True)

# 테이블의 긴 글(답, 조문)이 50자에서 잘리지 않고 끝까지 보이게 합니다
pd.set_option("display.max_colwidth", None)

print("생성용 LLM:", model_name)

In [ ]:
# 교안 01에서 만든 원문 정리, 생성 대상 고르기, 숫자 대조, 자카드, 토큰 합계, 금지 표현 검사입니다
import re
import unicodedata


def clean_text(text):
    """조문을 생성용 LLM의 입력이자 숫자 대조의 기준으로 쓸 수 있게 정리합니다.

    본문이 아닌 개정 태그는 지우고, 다르게 적힌 글자와 공백은 한 가지로 맞춥니다.
    """
    # NFC 정규화: 모양은 같지만 유니코드 번호가 다른 글자를 표준 번호 하나로 바꿉니다
    # 예: 조문의 '金'은 호환용 한자(U+F90A)라서, 답에 쓴 일반 한자 '金'(U+91D1)과 문자열 비교에서 다른 글자가 됩니다
    text = unicodedata.normalize("NFC", text)

    # 가운뎃점 자리에 쓴 한글 자모(U+318D)를 가운뎃점(U+00B7)으로 바꿉니다
    # 안 바꾸면 FAQ와 생성용 LLM의 답에 쓰인 가운뎃점과 다른 글자로 셉니다
    text = text.replace("\u318d", "\u00b7")

    # <개정 2012.2.1>, <신설 2008.3.21> 같은 개정 태그를 지웁니다. 시행일자는 교안 02의 데이터 카드에 적습니다
    # 안 지우면 날짜 2012, 2, 1 때문에 '2일' 같은 틀린 답이 숫자 대조를 통과합니다
    text = re.sub(r"<[^>]*>", "", text)

    # 줄바꿈과 겹친 공백을 공백 한 칸으로 맞춥니다(split()은 둘 다 구분자로 봅니다)
    # 안 맞추면 공백 모양만 다른 같은 문장을 다른 문장으로 봅니다
    return " ".join(text.split())


# 세부 사항을 시행령, 시행규칙 등에 넘기기만 하는 문장의 끝맺음
# 전자상거래법은 세부 사항을 총리령에도 넘기므로, 교안 01의 끝맺음에 총리령 끝맺음을 더했습니다
DELEGATION_ENDINGS = ("대통령령으로 정한다.", "고용노동부령으로 정한다.", "정하는 바에 따른다.", "총리령으로 정한다.")


def is_qa_target(text):
    """Q&A를 만들 문단이면 True, 위임만 하는 한 문장이거나 목적 조항이면 False를 돌려줍니다."""
    # 1) 한 문장(중간에 "다. "가 없음)이고 위임 끝맺음으로 끝나면 제외합니다
    #    endswith에 튜플을 주면 튜플 안의 끝맺음 가운데 하나로 끝나는지 봅니다
    if "다. " not in text and text.endswith(DELEGATION_ENDINGS):
        return False

    # 2) 조 제목이 (목적)인 목적 조항은 제외합니다
    #    (의\d+)?는 "제43조의2"처럼 가지 번호가 붙은 조도 맞추려는 부분입니다
    if re.match(r"^제\d+조(의\d+)?\(목적\)", text):
        return False
    return True


def numbers_match(answer, paragraph):
    """답에 나온 숫자가 모두 조문에도 있으면 True를 돌려줍니다."""
    # 조문의 숫자 리스트
    paragraph_numbers = re.findall(r"\d+", paragraph)

    # 답의 숫자 가운데 하나라도 조문에 없으면 False
    for number in re.findall(r"\d+", answer):
        if number not in paragraph_numbers:
            return False
    return True


def jaccard(a, b):
    """공백을 뺀 두 문장의 글자 집합으로 계산한 자카드 유사도를 돌려줍니다."""
    chars_a = set(a.replace(" ", ""))
    chars_b = set(b.replace(" ", ""))
    return len(chars_a & chars_b) / len(chars_a | chars_b)


def sum_total_tokens(usage_metadata):
    """모델 이름별 usage_metadata의 total_tokens를 모두 더해 돌려줍니다."""
    total = 0
    for model_usage in usage_metadata.values():
        total += model_usage["total_tokens"]
    return total


# 2번 프롬프트 규칙 2에서 막은 표현과, 7번 생성용 LLM이 일부만 답하고 붙이는 거절 문장('안내해 드릴 수 없', '제공된 조문')
BANNED_PHRASES = ["조문에 따르면", "조문상", "이 조문", "이 조항", "해당 조항", "해당 규정", "안내해 드릴 수 없", "제공된 조문"]


def has_banned_phrase(answer):
    """답에 막은 표현이 하나라도 있으면 True를 돌려줍니다."""
    for phrase in BANNED_PHRASES:
        if phrase in answer:
            return True
    return False

In [ ]:
# 교안 01에서 만든 비용 계산입니다
# gpt-5.6-luna 가격(달러, 토큰 100만 개당). 가격은 바뀌므로 OpenAI 가격 페이지에서 확인합니다
INPUT_PRICE = 0.20
CACHED_INPUT_PRICE = 0.02
OUTPUT_PRICE = 1.20


def usage_cost(usage_metadata):
    """모델 이름별 usage_metadata의 토큰 수에 가격을 곱한 비용(달러)을 돌려줍니다."""
    cost = 0
    for model_usage in usage_metadata.values():
        # 입력 토큰 가운데 캐시에서 읽은 토큰은 캐시 입력 가격, 나머지는 입력 가격으로 셉니다
        cached = model_usage.get("input_token_details", {}).get("cache_read", 0)
        cost += (model_usage["input_tokens"] - cached) * INPUT_PRICE
        cost += cached * CACHED_INPUT_PRICE

        # 출력 토큰(추론 토큰 포함)은 출력 가격으로 셉니다
        cost += model_usage["output_tokens"] * OUTPUT_PRICE
    return cost / 1000000

## 1. 조문 읽고 정리한 뒤 생성 대상 고르기

**배경**: 조문은 생성용 LLM의 입력이자 숫자 대조의 기준이므로, 교안 01처럼 개정 태그(`<개정 2016.3.29>`)를 지우고 줄바꿈과 가운뎃점 자리의 한글 자모 `ㆍ`를 맞춘 본문으로 Q&A를 만들되, 목적 조항과 세부 사항을 대통령령, 총리령에 넘기기만 하는 문단은 쓸모없는 답이 되므로 생성 대상에서 뺍니다.

**요구사항**
- (1) `data/과제/commerce_law_articles.json`을 읽어 문단 딕셔너리(dict)의 리스트 **`commerce_articles`** 에 담습니다.
- (2) 문단마다 `clean_text`로 정리한 본문을 `text`에 다시 넣고, 정리로 본문이 바뀐 문단 수를 정수 **`changed_count`** 에 담습니다.
- 셀 끝의 제공 코드가 주제 태그별 문단 수 `source_counts`를 만듭니다. 6번에서 씁니다.
- (3) 정리한 `commerce_articles`를 차례로 보며 `is_qa_target(text)`가 `True`인 문단을 **`qa_articles`** 에 담습니다. 3~4번의 Q&A 생성 입력에 씁니다.
- (4) `is_qa_target(text)`가 `False`인 문단은 **`skipped_articles`** 에 담습니다. 5번 숫자 대조와 7번 검색기에는 이 문단까지 들어 있는 `commerce_articles` 전체를 씁니다.

**예시**
- 제20조 제2항 원문의 숫자 리스트는 `['20', '2016', '3', '29', '2026', '1', '20']`입니다. 끝의 `<개정 2016.3.29, 2026.1.20>`을 지우면 `['20']`만 남습니다.
- 제20조 제3항은 "대통령령으로 정한다."로 끝나지만 앞에 통신판매중개자가 분쟁 해결에 필요한 조치를 해야 한다는 문장이 있어 `is_qa_target`이 `True`입니다.

**확인 기준**
- `commerce_articles`: 39문단이고 키가 `doc_id`, `law`, `source`, `text`입니다. 어느 문단의 `text`에도 `<`, 줄바꿈, 한글 자모 `ㆍ`가 없습니다.
- `changed_count`: 자가채점 셀에 적어 둔 기대값과 같습니다.
- `qa_articles`, `skipped_articles`: `commerce_articles`의 문단을 순서대로 둘로 나눈 리스트이고, 생성 대상 문단 수와 제외한 문단의 `doc_id`가 자가채점 셀에 적어 둔 기대값과 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01의 원문 정리와 생성 대상 고르기와 같은 순서입니다. 파일을 읽고, 문단마다 정리 전 본문과 정리한 본문을 비교해 다르면 하나 세고, 정리한 본문을 다시 넣은 뒤 생성 대상을 고릅니다.

세부구현:
1. JSON 파일은 data_dir 아래 과제 폴더에 있습니다. 파일 글자를 읽어 json.loads로 바꿉니다.
2. 비교는 text에 다시 넣기 전에 합니다. 넣은 뒤에 비교하면 늘 같아 0이 됩니다.
3. 고르기는 정리한 본문으로 합니다. 정리 전 원문은 끝에 개정, 신설 태그가 붙어 있으면 끝맺음 비교가 맞지 않습니다.
```

</details>

In [ ]:
# [작성] (1) 조문을 읽고, (2) 정리한 본문으로 바꾸며 바뀐 문단 수를 세세요.

# (1) 문단 딕셔너리 리스트 읽기
commerce_articles = ...

# (2) 문단마다 정리한 본문을 text에 다시 넣고, 바뀐 문단 수 세기
changed_count = 0
for article in commerce_articles:
    ...

print("문단 수:", len(commerce_articles), "| 정리로 바뀐 문단 수:", changed_count)

# 주제 태그별 문단 수(6번에서 씁니다)
commerce_table = pd.DataFrame(commerce_articles)
source_counts = commerce_table["source"].value_counts()
display(source_counts)

In [ ]:
# [작성] (3), (4) 정리한 본문으로 생성 대상 문단과 제외할 문단을 나누세요.
qa_articles = []
skipped_articles = []
for article in commerce_articles:
    ...

# 제외한 문단(정리한 본문 text 포함)
print("생성 대상 문단 수:", len(qa_articles), "| 제외한 문단 수:", len(skipped_articles))
display(pd.DataFrame(skipped_articles))

In [ ]:
# [자가채점] 문단 수, 정리한 본문, 바뀐 문단 수, 생성 대상 문단을 확인합니다.
assert isinstance(commerce_articles, list), "commerce_articles는 문단 딕셔너리의 리스트입니다. `...`를 채웠는지, 파일을 json.loads로 읽었는지 확인하세요."
assert len(commerce_articles) == 39, f"문단 수가 {len(commerce_articles)}입니다. data/과제/commerce_law_articles.json을 그대로 읽으면 39문단입니다."

for article in commerce_articles:
    assert list(article) == ["doc_id", "law", "source", "text"], "문단 딕셔너리의 키는 doc_id, law, source, text입니다. 키를 바꾸지 말고 text 값만 바꾸세요."
    assert "<" not in article["text"], f"{article['doc_id']}에 개정 태그(<개정 2016.3.29> 같은 꺾쇠 표시)가 남아 있습니다. clean_text로 정리한 본문을 text에 다시 넣으세요."
    assert "\n" not in article["text"], f"{article['doc_id']}에 줄바꿈이 남아 있습니다. clean_text로 정리한 본문을 text에 다시 넣으세요."
    assert "\u318d" not in article["text"], f"{article['doc_id']}에 한글 자모 가운뎃점이 남아 있습니다. clean_text로 정리한 본문을 text에 다시 넣으세요."

# 정리로 바뀐 문단 수(미리 구한 값)
expected_changed_count = 15
assert isinstance(changed_count, int), "changed_count는 정수입니다."
assert changed_count == expected_changed_count, f"정리로 바뀐 문단 수({changed_count})가 기대값과 다릅니다. text에 다시 넣기 전에 정리 전 본문과 정리한 본문을 비교했는지 확인하세요."

# 생성 대상 문단(미리 구한 값): 두 리스트가 commerce_articles를 빠짐없이 나누고, 제외한 문단이 기대값과 같아야 합니다
expected_qa_count = 37
expected_skipped_ids = ["전자상거래법 제1조", "전자상거래법 제45조 제8항"]
assert isinstance(qa_articles, list) and isinstance(skipped_articles, list), "qa_articles, skipped_articles는 문단 딕셔너리의 리스트입니다. `...`를 채웠는지 확인하세요."
assert len(qa_articles) + len(skipped_articles) == len(commerce_articles), "문단마다 qa_articles와 skipped_articles 가운데 한 곳에만 넣으세요."
skipped_ids = []
for article in skipped_articles:
    skipped_ids.append(article["doc_id"])
assert skipped_ids == expected_skipped_ids, f"제외한 문단이 {skipped_ids}입니다. 정리한 본문(text)으로 is_qa_target을 불렀는지 확인하세요. 정리 전 원문은 끝에 붙은 태그 때문에 위임 끝맺음 비교가 맞지 않습니다."
assert len(qa_articles) == expected_qa_count, f"생성 대상 문단이 {len(qa_articles)}개입니다. 정리한 본문(text)으로 is_qa_target을 불렀는지 확인하세요."
for article in qa_articles:
    assert article in commerce_articles, "qa_articles에는 commerce_articles의 문단 딕셔너리를 그대로 담습니다."
    assert is_qa_target(article["text"]), f"{article['doc_id']}는 정리한 본문의 is_qa_target이 False인데 qa_articles에 있습니다. 정리한 본문(text)으로 고르세요."
print("✅ 통과!")

**결과 읽기**
- **정리로 바뀐 15문단:** 태그가 있던 11문단은 정리 전 숫자 리스트에 날짜가 섞여 있어, 정리하지 않으면 숫자 대조가 날짜까지 근거로 셉니다.
- **제외한 2문단:** 목적 조항인 제1조와 위임 한 문장뿐인 제45조 제8항입니다. 둘 다 거래조건 표시·청약확인 태그라, 4번에서 이 태그만 `source_counts`(9문단)와 `qa_source_counts`(7문단)가 다릅니다.
- **끝맺음 튜플:** 다른 법령에 옮겨 쓸 때는 그 법이 위임하는 곳의 끝맺음(여기서는 총리령)을 `DELEGATION_ENDINGS`에 더해야 위임 문단이 걸러집니다.

## 2. Q&A 생성 체인 만들기

**배경**: 조문 하나에서 받은 Q&A 여러 건을 행으로 모으고 질문 유형 분포를 보려면, 생성용 LLM이 질문, 답, 질문 유형을 정해진 칸에 담은 리스트로 돌려줘야 합니다.

**요구사항**
- (1) Q&A 한 건을 담는 Pydantic 모델 **`QAPair`** 를 만듭니다. 필드는 문자열 `question`, `answer`와 질문 유형 `question_type`(타입 `Literal["사실 확인", "절차", "문제 상황"]`)입니다. 필드 이름과 description(모델이 보는 칸 설명)은 셀에 적어 두었으니 `...` 자리에 타입만 채웁니다.
- (2) 조문 하나에서 만든 Q&A 리스트를 담는 **`QASet`**(필드 `items`, 타입은 `list[QAPair]`)을 같은 방법으로 만듭니다.
- (3) system 메시지 "온라인 쇼핑 상담 챗봇이 배울 질문과 답을 써야 한다. 네가 쓴 질문과 답은 그대로 챗봇의 학습 예시가 된다."와 셀에 적어 둔 human 메시지 `qa_human`으로 프롬프트 템플릿 **`qa_prompt`** 를 만듭니다.
- (4) `qa_prompt`와 `QASet` 모양으로만 답하는 `llm`을 이어 체인 **`qa_generator`** 를 만듭니다.
- (5) 모델을 부르지 않고 `qa_prompt`의 `{n}`에 3, `{paragraph}`에 제17조 제1항의 정리한 본문(셀의 `article_by_id`로 찾은 문단의 `text`)을 채운 메시지 리스트를 **`preview_messages`** 에 담습니다.

**예시**: `preview_messages`의 human 메시지는 "아래 [조문]으로, 온라인 쇼핑 이용자나 판매자가 상담 챗봇에 실제로 물어볼 Q&A를 1~3개 만들어라."로 시작하고, `[조문]` 아래에 "제17조(청약철회등) 통신판매업자와 재화등의 구매에 관한 계약을 체결한 소비자는 …"이 들어 있습니다.

**확인 기준**
- `QAPair`, `QASet`: `QAPair`의 필드가 `question`, `answer`, `question_type`이고 `question_type`이 세 값의 `Literal`입니다. `QASet`의 필드는 `items`(`list[QAPair]`)입니다.
- `qa_prompt`, `qa_generator`: 빈자리가 `n`, `paragraph` 두 개이고, 체인이 `qa_prompt`로 시작해 `QASet`을 돌려줍니다.
- `preview_messages`: system, human 순서의 메시지 두 개이고, human 메시지에 "1~3개"와 제17조 제1항의 정리한 본문이 들어 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01의 Q&A 생성 체인과 같은 모양입니다. 스키마, 프롬프트 템플릿, 구조화 출력을 차례로 만들고 |로 잇습니다.

세부구현:
1. 리스트 필드는 리스트 타입에 원소의 모델 이름을 넣어 적습니다. 정해진 값 가운데 하나만 받는 필드는 typing의 Literal에 값을 나열합니다.
2. 프롬프트 템플릿은 (역할, 문장) 튜플 두 개의 리스트로 만듭니다.
3. 모델 호출 없이 빈자리를 채운 메시지를 받는 메서드는 format_messages입니다. 정리한 본문은 doc_id로 찾은 문단의 text입니다.
```

</details>

In [ ]:
# [작성] (1), (2) 필드마다 ... 자리에 타입을 채워 Q&A 한 건과 Q&A 리스트의 모양을 정하세요.
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class QAPair(BaseModel):
    """조문에 근거한 Q&A 한 건입니다."""
    question: ... = Field(description="조문 내용만으로 답할 수 있는 온라인 쇼핑 이용자나 판매자의 질문")
    answer: ... = Field(description="조문에 적힌 내용으로 쓴 상담원의 한두 문장 답")
    question_type: ... = Field(
        description="질문 유형. 사실 확인은 기준, 기간, 금액이 얼마인지 묻는 질문, 절차는 무엇을 언제 어떻게 해야 하는지 묻는 질문, 문제 상황은 판매자가 이렇게 했는데 괜찮은지나 내가 받을 수 있는지 묻는 질문"
    )


class QASet(BaseModel):
    """조문 하나에서 만든 Q&A 리스트입니다."""
    items: ... = Field(description="서로 다른 Q&A 리스트")

In [ ]:
# [작성] (3), (4) 프롬프트 템플릿과 체인을 만들고, (5) 모델 호출 없이 보낼 메시지를 미리 보세요.

# human 메시지: 규칙과 조문. {n}(만들 개수의 상한)과 {paragraph}(조문 본문)는 부를 때 채웁니다
qa_human = """아래 [조문]으로, 온라인 쇼핑 이용자나 판매자가 상담 챗봇에 실제로 물어볼 Q&A를 1~{n}개 만들어라. 조문에서 물을 것이 적으면 적게 만든다.

근거:
1. [조문]에 적힌 내용만 쓰고, 조문에 없는 내용은 지어내지 않는다.

답 쓰는 법:
2. 답은 상담원이 직접 하는 존댓말로 한두 문장 쓰고, '조문에 따르면'·'조문상'·'이 조문'·'이 조항'·'해당 조항'·'해당 규정' 같은 말은 쓰지 않는다.
3. 답에는 조문에 적힌 조건(대상, 기간, 비율, 예외)을 빠뜨리지 않는다.

질문 쓰는 법:
4. 질문마다 유형(사실 확인, 절차, 문제 상황)을 정하고, 여러 개를 만들면 유형을 섞는다.
5. 같은 답이 나오는 질문은 하나만 만든다.
6. 질문은 법률 용어 대신 묻는 사람이 평소 쓰는 말로, 자기 상황을 말하며 묻는 문장으로 쓴다.

조·항 번호:
7. 질문에는 조·항 번호(제○조, 제○항)를 쓰지 않는다. 답에서 조문을 밝힐 때는 '전자상거래법 제17조'처럼 법령 이름과 조 번호만 쓰고, 항·호 번호(제○항, 제○호)는 쓰지 않는다.

[조문]
{paragraph}"""

# (3) system은 역할, human은 위 규칙과 조문인 프롬프트 템플릿
qa_prompt = ...

# (4) 프롬프트 -> 생성용 LLM(QASet 모양으로만 답함) 체인
qa_generator = ...

# (5) 제17조 제1항으로 채운 메시지(모델 호출 없음)
article_by_id = {article["doc_id"]: article for article in commerce_articles}
preview_messages = ...
for message in preview_messages:
    print(f"[{message.type}]")
    print(message.content)

In [ ]:
# [자가채점] 스키마, 프롬프트 템플릿, 체인, 미리 본 메시지를 확인합니다.
from typing import Literal

from langchain_core.runnables import RunnableSequence

assert list(QAPair.model_fields) == ["question", "answer", "question_type"], "QAPair의 필드는 question, answer, question_type 세 개이고 이 순서입니다. `...`를 채웠는지 확인하세요."
assert QAPair.model_fields["question_type"].annotation == Literal["사실 확인", "절차", "문제 상황"], "question_type의 타입은 Literal[\"사실 확인\", \"절차\", \"문제 상황\"]입니다. 세 값의 글자를 지문과 같게 적으세요."
assert list(QASet.model_fields) == ["items"], "QASet의 필드는 items 하나입니다."
assert QASet.model_fields["items"].annotation == list[QAPair], "QASet의 items 타입은 list[QAPair]로 적습니다(typing.List가 아니라 소문자 list)."

assert isinstance(qa_prompt, ChatPromptTemplate), "qa_prompt는 ChatPromptTemplate.from_messages로 만든 프롬프트 템플릿입니다."
assert set(qa_prompt.input_variables) == {"n", "paragraph"}, f"qa_prompt의 빈자리는 n, paragraph 두 개여야 합니다. 지금: {qa_prompt.input_variables}. human 메시지로 qa_human을 넣었는지 확인하세요."
assert isinstance(qa_generator, RunnableSequence) and qa_generator.first is qa_prompt, "qa_generator는 qa_prompt로 시작하는 체인입니다. qa_prompt와 구조화 출력 llm을 |로 이으세요."
assert qa_generator.output_schema.model_json_schema()["title"] == "QASet", "qa_generator가 QASet을 돌려주지 않습니다. llm에 QASet으로 구조화 출력을 거세요."

# 미리 본 메시지: system, human 두 개와 채운 값
assert isinstance(preview_messages, list) and len(preview_messages) == 2, "preview_messages는 메시지 두 개(system, human)의 리스트입니다. qa_prompt의 format_messages로 만드세요."
assert preview_messages[0].type == "system" and preview_messages[1].type == "human", "메시지 순서는 system, human입니다."
assert preview_messages[0].content == "온라인 쇼핑 상담 챗봇이 배울 질문과 답을 써야 한다. 네가 쓴 질문과 답은 그대로 챗봇의 학습 예시가 된다.", "system 메시지는 지문의 문장 그대로입니다."
assert "Q&A를 1~3개" in preview_messages[1].content, "human 메시지의 {n} 자리에 3을 넣으세요. qa_human을 바꾸지 마세요."
assert article_by_id["전자상거래법 제17조 제1항"]["text"] in preview_messages[1].content, "human 메시지의 {paragraph} 자리에 제17조 제1항의 정리한 본문(text)을 넣으세요."
print("✅ 통과!")

**결과 읽기**
- **`[human]`:** `{n}`, `{paragraph}` 자리에 3과 정리한 제17조 제1항이 들어갔는지, 개정 태그가 남지 않았는지 봅니다.
- **모델 호출:** 아직 없습니다. `qa_generator`는 3번에서 처음 부릅니다.

## 3. 주제가 다른 5개 문단으로 먼저 시험하기

**배경**: 생성 대상 문단 전체를 보내기 전에 주제 태그가 서로 다른 5개 문단으로 결과 모양과 토큰 수를 확인하고, 교안 01처럼 시험에 든 비용으로 생성 대상 전체 문단의 비용을 어림합니다.

**요구사항**
- (1) `qa_articles`를 앞에서부터 보며 주제 태그가 처음 나온 문단을 하나씩 모아, 태그가 서로 다른 문단 5개를 **`trial_articles`** 에 담습니다.
- (2) 5개 문단마다 `{"n": 3, "paragraph": 본문}` 입력을 만들고, `qa_generator.batch`에 `config={"max_concurrency": MAX_CONCURRENCY}`와 `return_exceptions=True`를 주어 한꺼번에 보낸 결과 리스트를 **`trial_results`** 에 담습니다. 실패한 요청은 결과 자리에 예외 객체가 들어갑니다. 이 호출은 `get_usage_metadata_callback()`의 `with` 블록(`as usage`) 안에서 합니다.
- (3) `usage.usage_metadata`를 `sum_total_tokens`로 더한 5개 문단의 토큰 수를 정수 **`trial_tokens`** 에 담습니다.
- (4) `trial_tokens`를 5로 나누고 생성 대상 문단 수를 곱해 `round`로 반올림한 정수를 **`estimated_tokens`** 에 담습니다.
- (5) with 블록이 끝난 뒤 `usage.usage_metadata`를 단계별 사용량 딕셔너리 **`stage_usage`** 에 `"5개 문단 시험"` 키로 남깁니다. 4, 6, 7, 8번도 요청을 보낸 단계마다 같은 방법으로 남기고, 10번에서 실제 비용을 계산합니다.
- (6) `usage_cost(stage_usage["5개 문단 시험"])`를 시험 문단 수(`len(trial_inputs)`)로 나눈 문단 하나의 평균 비용(달러)을 **`cost_per_paragraph`** 에 담습니다.
- (7) `cost_per_paragraph`에 생성 대상 문단 수(`len(qa_articles)`)를 곱한 전체 문단 예상 비용(달러)을 **`estimated_cost`** 에 담습니다. 4번에서 실제 비용과 비교합니다.

**예시**
- 5개 문단에 2,000토큰이 들었다면 생성 대상 37문단은 2000 / 5 x 37 = 14,800토큰으로 어림합니다.
- 시험 5문단에 0.002달러가 들었다면 문단당 평균 비용은 0.0004달러이고, 생성 대상 37문단의 예상 비용은 0.0004 x 37 = 0.0148달러입니다.

**확인 기준**
- 시험: `trial_articles`가 `qa_articles`의 문단 5개이고 주제 태그가 모두 다르며, doc_id가 자가채점 셀에 적어 둔 순서와 같습니다. `trial_results`가 5개이고, 예외 객체가 아닌 결과는 `QASet`이며 Q&A를 1건 이상 받은 결과가 하나 이상입니다.
- 토큰 수: `trial_tokens`가 0보다 큰 정수이고, `estimated_tokens`가 `trial_tokens`의 7배보다 크고 8배보다 작은 정수입니다. `stage_usage`에 `"5개 문단 시험"` 키가 있고 그 토큰 수가 `trial_tokens`와 같습니다.
- 비용: `cost_per_paragraph`가 0보다 크고 `usage_cost(stage_usage["5개 문단 시험"]) / len(trial_inputs)`와 같으며, `estimated_cost`가 `cost_per_paragraph * len(qa_articles)`와 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01에서 주제가 다른 5개 문단을 골라 먼저 시험하고 예상 비용을 어림할 때와 같은 방법입니다. 태그를 본 리스트를 두고, 처음 보는 태그의 문단만 담다가 5개가 되면 멈춥니다.
- 비용은 제공 함수 usage_cost가 계산하므로, 시험 문단 수로 나누고 생성 대상 문단 수를 곱하기만 합니다.

세부구현:
1. 입력 딕셔너리 리스트는 trial_articles를 돌며 만듭니다.
2. with 블록이 끝난 뒤 usage.usage_metadata에 모델 이름별 토큰 사용량이 들어 있습니다. 제공 함수가 total_tokens를 모두 더해 줍니다.
3. usage_cost에 stage_usage["5개 문단 시험"]을 넣으면 시험 5문단의 비용(달러)이 나옵니다. 생성 대상 문단 수는 qa_articles의 길이입니다.
```

</details>

In [ ]:
# [작성] 주제 태그가 다른 5개 문단으로 먼저 시험하고 전체 토큰 수와 비용을 어림하세요.
from langchain_core.callbacks import get_usage_metadata_callback

# .batch가 동시에 보내는 요청 수의 상한입니다. 429 오류(요청 한도 초과)가 나면 줄입니다
MAX_CONCURRENCY = 32

# (1) 주제 태그가 처음 나온 문단을 하나씩, 5개 모으기
trial_articles = []
seen_sources = []
for article in qa_articles:
    ...

# 시험할 5개 문단(체인에 넣을 본문 text 포함)
display(pd.DataFrame(trial_articles))

# (2) 5개 문단의 입력을 만들어 한꺼번에 보내기(토큰 사용량을 usage에 기록)
trial_inputs = ...
with get_usage_metadata_callback() as usage:
    trial_results = ...

# (3) 5개 문단의 토큰 수와 (4) 생성 대상 전체의 어림값(실패한 요청은 사용량이 기록되지 않아, 실패가 있으면 어림값이 작게 나옵니다)
trial_tokens = ...
estimated_tokens = ...
print("5개 문단 토큰 수:", trial_tokens, "| 전체 예상 토큰 수:", estimated_tokens)

# API를 부른 단계마다 토큰 사용량을 모으는 딕셔너리(키: 단계 이름). 10번에서 실제 비용을 계산할 때 씁니다
stage_usage = {}

# (5) with 블록의 토큰 사용량(usage.usage_metadata)을 "5개 문단 시험" 키로 stage_usage에 넣기
...

# (6) 시험 5문단의 비용(usage_cost)을 시험 문단 수로 나눈 문단 하나의 평균 비용(달러)
cost_per_paragraph = ...

# (7) 문단당 평균 비용 x 생성 대상 문단 수
estimated_cost = ...
print("문단당 평균 비용(달러):", round(cost_per_paragraph, 5), "| 전체 문단 예상 비용(달러):", round(estimated_cost, 4))

In [ ]:
# 5개 문단의 Q&A를 doc_id와 함께 테이블로 봅니다(실패한 요청과 Q&A가 하나도 없는 결과는 건너뛰고 셉니다)
trial_rows = []
trial_failed = 0
for article, result in zip(trial_articles, trial_results):
    if isinstance(result, Exception) or len(result.items) == 0:  # 빈 결과도 실패로 셉니다
        trial_failed += 1
        continue
    for item in result.items:
        trial_rows.append({"doc_id": article["doc_id"], "question_type": item.question_type, "question": item.question, "answer": item.answer})
print("실패한 요청 수:", trial_failed)
display(pd.DataFrame(trial_rows))

In [ ]:
# [자가채점] 시험한 문단, 결과 모양, 토큰 수를 확인합니다.
expected_trial_ids = ["전자상거래법 제13조 제1항", "전자상거래법 제15조 제1항", "전자상거래법 제17조 제1항", "전자상거래법 제18조 제1항", "전자상거래법 제20조 제1항"]

assert isinstance(trial_articles, list) and len(trial_articles) == 5, "trial_articles는 문단 5개의 리스트입니다. 5개를 모으면 반복을 멈추세요."
trial_ids = []
trial_sources = []
for article in trial_articles:
    assert article in qa_articles, "trial_articles에는 qa_articles의 문단 딕셔너리를 그대로 담습니다."
    trial_ids.append(article["doc_id"])
    trial_sources.append(article["source"])
assert len(set(trial_sources)) == 5, f"주제 태그가 서로 달라야 합니다. 지금 태그: {trial_sources}"
assert trial_ids == expected_trial_ids, f"trial_articles가 {trial_ids}입니다. 앞에서부터 보며 태그가 처음 나온 문단을 담았는지 확인하세요."

assert isinstance(trial_results, list) and len(trial_results) == 5, "trial_results는 5개 문단의 결과 리스트입니다. .batch에 입력 5개를 넣었는지 확인하세요."
# Q&A를 1건 이상 받은 결과 수(예외 객체와 빈 결과는 실패로 셉니다)
trial_success = 0
for result in trial_results:
    if isinstance(result, Exception):
        continue
    assert isinstance(result, QASet), "성공한 결과는 QASet이어야 합니다. qa_generator.batch로 보냈는지 확인하세요."
    if len(result.items) >= 1:
        trial_success += 1
assert trial_success >= 1, "5개 요청 가운데 Q&A를 받은 결과가 없습니다. 오류 메시지(trial_results의 예외 객체)를 보고 위 [작성] 셀을 다시 실행하세요. 빈 결과만 왔다면 입력 딕셔너리의 n에 3을 넣었는지 확인하세요."
assert isinstance(trial_tokens, int) and trial_tokens > 0, "trial_tokens는 0보다 큰 정수입니다. .batch 호출을 get_usage_metadata_callback()의 with 블록 안에 두고, usage.usage_metadata를 sum_total_tokens로 더하세요."
assert isinstance(estimated_tokens, int), "estimated_tokens는 정수입니다. round로 반올림하세요."
assert trial_tokens * 7 < estimated_tokens < trial_tokens * 8, f"estimated_tokens({estimated_tokens})는 trial_tokens를 5로 나누고 생성 대상 문단 수 {len(qa_articles)}를 곱한 값(약 7.4배)입니다."

# 단계별 사용량: 5개 문단 시험의 usage_metadata
assert isinstance(stage_usage, dict) and "5개 문단 시험" in stage_usage, "stage_usage에 \"5개 문단 시험\" 키가 없습니다. with 블록이 끝난 뒤 usage.usage_metadata를 이 키로 넣으세요."
assert sum_total_tokens(stage_usage["5개 문단 시험"]) == trial_tokens, "stage_usage[\"5개 문단 시험\"]의 토큰 수가 trial_tokens와 다릅니다. 5개 문단을 보낸 with 블록의 usage.usage_metadata를 그대로 넣으세요."

# 비용: 문단당 평균 비용과 전체 문단 예상 비용
trial_cost = usage_cost(stage_usage["5개 문단 시험"])
assert isinstance(cost_per_paragraph, float) and cost_per_paragraph > 0, "cost_per_paragraph는 0보다 큰 실수입니다. usage_cost(stage_usage[\"5개 문단 시험\"])를 len(trial_inputs)로 나누세요."
assert abs(cost_per_paragraph - trial_cost / len(trial_inputs)) < 1e-12, f"cost_per_paragraph가 {cost_per_paragraph}입니다. 시험 비용 {trial_cost}를 시험 문단 수 {len(trial_inputs)}로 나눈 값이어야 합니다."
assert abs(estimated_cost - cost_per_paragraph * len(qa_articles)) < 1e-9, f"estimated_cost가 {estimated_cost}입니다. cost_per_paragraph에 생성 대상 문단 수 {len(qa_articles)}를 곱하세요."
print("✅ 통과!")

**결과 읽기**
- **테이블:** 질문과 답이 `qa_human`의 규칙(존댓말 한두 문장, 조건 유지, 유형 섞임, 조·항 번호)을 지키는지 읽고, 어긋나면 규칙을 고쳐 다시 시험합니다.
- **어림값:** 시험 5문단의 평균이라 실제와 다를 수 있고, 실패한 요청은 사용량이 기록되지 않아 실패가 있으면 작게 어림합니다.
- **뒤 단계 비용:** 6, 7, 8번은 요청 수가 앞 단계 결과로 정해져 미리 어림하지 않고, 10번에서 실제 비용만 봅니다.

## 4. 생성 대상 37문단 전체로 Q&A 만들기

**배경**: 요청이 많으면 일부가 시간 초과나 요청 한도 초과로 실패할 수 있으므로, 실패한 요청은 따로 모으고 성공한 결과만 행으로 펼친 뒤 주제, 질문 유형, 문단당 Q&A 수가 한쪽으로 치우치지 않았는지 봅니다.

**요구사항**
- (1) 생성 대상 문단(`qa_articles`) 전체의 입력(`n=3`)을 `qa_generator.batch`로 보내되, `max_concurrency`와 함께 `return_exceptions=True`를 주어 결과를 **`qa_results`** 에 담습니다. 요청은 이 셀에서만 보내므로, 뒤 셀의 반복문을 고칠 때 요청을 다시 보내지 않습니다.
  - 이 호출은 `get_usage_metadata_callback()`의 `with` 블록(`as usage`) 안에서 하고, 블록이 끝난 뒤 `usage.usage_metadata`를 `stage_usage`에 `"조문으로 Q&A 생성"` 키로 남깁니다.
- (2) 성공한 결과의 Q&A를 한 건씩 행 딕셔너리로 펼쳐 **`qa_rows`** 에 담습니다. 키는 `id`, `doc_id`, `source`, `method`, `question`, `answer`, `question_type` 순서이고, `doc_id`와 `source`는 결과와 짝지은 문단의 값, `question_type`은 Q&A의 유형입니다. `id`는 `"cqa-0001"`부터 네 자리 번호, `method`는 `"문서 생성"`입니다.
- (3) 실패한 요청(예외 객체)과 Q&A가 하나도 없는 결과(`items`가 빈 리스트)는 `{"doc_id": 문단의 doc_id, "error": repr(결과)}`로 **`failed_rows`** 에 담고 다음 결과로 넘어갑니다.
  - 교안 01은 예외 객체만 실패로 셉니다. 빈 결과도 그 문단의 행을 하나도 만들지 못해 실패와 같으므로, 여기서는 함께 모아 다시 보낼 문단으로 둡니다.
- (4) 생성 대상 문단의 주제 태그별 문단 수(셀의 `qa_source_counts`)와 `qa_rows`의 주제 태그별 행 수를 나란히 놓은 데이터프레임(DataFrame) **`topic_coverage`**(열 `"문단 수"`, `"Q&A 행 수"`, 행 이름은 주제 태그)를 만듭니다.
- 아래 제공 셀이 질문 유형별 행 수 `type_counts`와 문단당 Q&A 수 `per_article_counts`(Q&A가 n개인 문단이 몇 개인지)를 보여 줍니다.

**예시**: 첫 문단에서 Q&A 2건이 오면 `cqa-0001`, `cqa-0002` 두 행이 같은 `doc_id` "전자상거래법 제13조 제1항"을 가지고, 다음 문단의 첫 행은 `cqa-0003`입니다.

**확인 기준**
- `qa_results`, `failed_rows`: `qa_results`가 `qa_articles`와 같은 37개이고, `stage_usage`에 `"조문으로 Q&A 생성"` 키가 있으며, `failed_rows`가 예외 객체이거나 Q&A가 없는 결과의 문단을 순서대로 하나씩 담았습니다.
- `qa_rows`: 행마다 키 일곱 개가 위 순서이고 `id`가 `cqa-0001`부터 차례대로입니다. `method`가 `"문서 생성"`, `question_type`이 세 유형 가운데 하나이고 `doc_id`, `source`가 `qa_articles`의 같은 문단 값이며, 행 수가 성공한 결과의 Q&A 수를 모두 더한 값과 같습니다.
- `topic_coverage`: 주제 태그 5행이고 `"문단 수"` 열이 `qa_source_counts`와 같으며, `"Q&A 행 수"` 열의 합이 `len(qa_rows)`입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01의 전체 생성과 같은 방법입니다. 문단과 결과를 zip으로 짝지어, 실패한 결과면 failed_rows로, 아니면 items를 돌며 qa_rows에 한 행씩 더합니다.

세부구현:
1. 결과가 실패인지는 isinstance로 Exception인지 봅니다. 예외 객체가 아니어도 items의 길이가 0이면 실패로 셉니다.
2. id 번호는 지금까지 모은 행 수에 1을 더한 값입니다. f-string에서 :04d로 네 자리를 0으로 채웁니다.
3. 주제 태그별 행 수는 qa_rows를 테이블로 바꾼 뒤 source 열의 value_counts로 셉니다. 두 Series를 딕셔너리 하나로 묶어 데이터프레임에 넣으면 같은 태그끼리 나란히 놓입니다.
4. with 블록과 stage_usage에 넣는 줄은 3번 시험 셀과 같은 모양이고, 단계 이름만 다릅니다.
```

</details>

In [ ]:
# [작성] (1) 생성 대상 문단 전체로 Q&A를 요청하고 토큰 사용량을 남기세요. 요청은 이 셀에서만 보냅니다.

# 생성 대상 문단의 입력
qa_inputs = ...

# get_usage_metadata_callback()의 with 블록(as usage) 안에서 qa_inputs를 한꺼번에 보내 qa_results에 담기(실패한 요청은 결과 자리에 예외 객체로)
...

# with 블록이 끝난 뒤 usage.usage_metadata를 "조문으로 Q&A 생성" 키로 stage_usage에 넣기
...

# 3번에서 어림한 값과 실제 값을 나란히 봅니다
print("받은 결과 수:", len(qa_results))
print("토큰 수 예상:", estimated_tokens, "| 실제:", sum_total_tokens(usage.usage_metadata))
print("비용(달러) 예상:", round(estimated_cost, 4), "| 실제:", round(usage_cost(usage.usage_metadata), 4))  # 실제: 이 단계 사용량 x 가격

In [ ]:
# [작성] (2), (3) 받은 결과를 행으로 펼치고 실패한 요청을 모으세요(이 셀을 다시 실행해도 요청은 다시 보내지 않습니다).
qa_rows = []
failed_rows = []
for article, result in zip(qa_articles, qa_results):
    ...

print("Q&A 행 수:", len(qa_rows), "| 실패한 요청 수:", len(failed_rows))
qa_table = pd.DataFrame(qa_rows)
display(qa_table.head())

In [ ]:
# [작성] (4) 주제 태그별 문단 수와 Q&A 행 수를 나란히 놓으세요.

# 생성 대상 문단의 주제 태그별 문단 수
qa_source_counts = pd.DataFrame(qa_articles)["source"].value_counts()  # 제외한 문단은 세지 않음

topic_coverage = ...
display(topic_coverage)

In [ ]:
# 질문 유형별 행 수
type_counts = qa_table["question_type"].value_counts()
display(type_counts)

# doc_id별 행 수를 한 번 더 세면 "Q&A가 n개인 문단이 몇 개인지"가 됩니다(n이 작은 순서)
qa_per_article = qa_table["doc_id"].value_counts()
per_article_counts = qa_per_article.value_counts().sort_index()
display(per_article_counts)

In [ ]:
# [자가채점] 결과 수, 실패한 요청, 행의 키와 번호, 주제별 테이블을 확인합니다.
assert isinstance(qa_results, list) and len(qa_results) == len(qa_articles), f"qa_results는 생성 대상 문단마다 하나씩, {len(qa_articles)}개입니다. qa_articles로 입력을 만들고 return_exceptions=True를 주었는지 확인하세요."
assert "조문으로 Q&A 생성" in stage_usage, "stage_usage에 \"조문으로 Q&A 생성\" 키가 없습니다. 요청 셀에서 with 블록이 끝난 뒤 usage.usage_metadata를 이 키로 넣으세요."

# 결과마다 실패(예외 객체나 Q&A가 없는 결과)인 문단과, 성공한 결과의 Q&A 수를 셉니다
expected_failed_ids = []
expected_row_count = 0
for article, result in zip(qa_articles, qa_results):
    if isinstance(result, Exception):
        expected_failed_ids.append(article["doc_id"])
        continue
    assert isinstance(result, QASet), "성공한 결과는 QASet이어야 합니다. qa_generator.batch로 보냈는지 확인하세요."
    if len(result.items) == 0:
        expected_failed_ids.append(article["doc_id"])
    expected_row_count += len(result.items)

# 실패한 요청: 그 문단의 doc_id를 순서대로 한 줄씩
failed_ids = []
for row in failed_rows:
    assert list(row) == ["doc_id", "error"], "failed_rows의 딕셔너리는 doc_id, error 두 키입니다."
    failed_ids.append(row["doc_id"])
assert failed_ids == expected_failed_ids, f"failed_rows의 doc_id가 {failed_ids}입니다. 예외 객체인 결과와 Q&A가 하나도 없는 결과마다 그 문단을 순서대로 한 줄씩 담으세요."

# 행 수는 성공한 결과의 Q&A 수를 모두 더한 값
assert len(qa_rows) == expected_row_count, f"Q&A 행 수({len(qa_rows)})가 성공한 결과의 Q&A 수 합({expected_row_count})과 다릅니다. 결과마다 items를 한 건도 빠뜨리지 않고 한 행씩 펼쳤는지 확인하세요."

# 행의 키, id 번호, method, 질문 유형, 근거 조문과 주제 태그
source_by_id = {article["doc_id"]: article["source"] for article in qa_articles}
for number, row in enumerate(qa_rows, start=1):
    assert list(row) == ["id", "doc_id", "source", "method", "question", "answer", "question_type"], "qa_rows의 행 키는 id, doc_id, source, method, question, answer, question_type 순서입니다."
    assert row["id"] == f"cqa-{number:04d}", f"{number}번째 행의 id가 {row['id']}입니다. cqa-0001부터 네 자리 번호로 차례대로 붙이세요."
    assert row["method"] == "문서 생성", "method는 \"문서 생성\"입니다."
    assert row["question_type"] in ["사실 확인", "절차", "문제 상황"], f"{row['id']}의 question_type({row['question_type']})이 세 유형이 아닙니다. QAPair의 question_type을 넣으세요."
    assert row["doc_id"] in source_by_id, f"{row['id']}의 doc_id({row['doc_id']})가 생성 대상 문단에 없습니다. 결과와 짝지은 문단의 doc_id를 넣으세요."
    assert row["source"] == source_by_id[row["doc_id"]], f"{row['id']}의 source({row['source']})가 {row['doc_id']}의 주제 태그와 다릅니다. 결과와 짝지은 문단의 source를 넣으세요."
    assert isinstance(row["question"], str) and isinstance(row["answer"], str), "question, answer에는 QAPair의 문자열을 넣습니다."

# 주제 태그별 테이블: 문단 수는 생성 대상 문단만 센 qa_source_counts
assert isinstance(topic_coverage, pd.DataFrame), "topic_coverage는 데이터프레임입니다."
assert list(topic_coverage.columns) == ["문단 수", "Q&A 행 수"], "topic_coverage의 열은 \"문단 수\", \"Q&A 행 수\" 순서입니다."
assert set(topic_coverage.index) == set(qa_source_counts.index), "topic_coverage의 행 이름은 주제 태그 5개입니다."
for tag in qa_source_counts.index:
    assert topic_coverage.loc[tag, "문단 수"] == qa_source_counts[tag], f"{tag}의 \"문단 수\"가 {topic_coverage.loc[tag, '문단 수']}입니다. 생성 대상 문단만 센 qa_source_counts를 넣으세요. source_counts는 제외한 문단까지 센 값입니다."
assert topic_coverage["Q&A 행 수"].sum() == len(qa_rows), "\"Q&A 행 수\" 열의 합이 len(qa_rows)와 다릅니다. qa_rows의 주제 태그별 행 수를 넣으세요."
print("✅ 통과!")

**결과 읽기**
- **예상과 실제:** 크게 어긋나면 5개 문단이 전체를 대표하지 못한 것입니다. 실패한 문단만 다시 보내려면 `failed_rows`의 문단 입력만 모아 `.batch`로 보냅니다.
- **주제별 행 수:** 문단이 4개인 공급·배송은 행도 적어, 6번 질문 늘리기가 모자란 만큼 채웁니다.
- **유형과 문단당 Q&A 수:** 유형이 한쪽으로 몰리면 `qa_human`의 규칙 4를 고칩니다. 모든 문단이 3개면 억지로 채웠는지, 대부분 1개면 조건을 빠뜨렸는지 읽어 봅니다.

## 5. 숫자 대조와 금지 표현으로 검수하기

**배경**: 온라인 쇼핑 상담의 답은 7일, 3영업일, 3개월 같은 기한이 핵심이라, 답의 숫자가 근거 조문에 있는지와 프롬프트로 막은 표현이 남았는지를 함께 검사합니다.

**요구사항**
- (1) `doc_id`로 정리한 본문을 찾는 딕셔너리 **`doc_texts`**(키 `doc_id`, 값 `text`)를 `commerce_articles` 전체로 만듭니다. 7번 생성용 LLM은 생성 대상에서 뺀 문단도 근거로 고를 수 있어 그 문단도 넣습니다.
- (2) `qa_rows` 가운데 `numbers_match(답, 근거 조문)`이 `True`이고 `has_banned_phrase(답)`이 `False`인 행을 순서대로 **`passed_rows`** 에 담습니다.
- (3) 나머지 행마다 키가 `"id"`, `"doc_id"`, `"answer"`, `"숫자 대조"`(`numbers_match` 결과), `"금지 표현"`(`has_banned_phrase` 결과) 순서인 딕셔너리를 **`rejected_rows`** 에 담습니다. 키는 테이블의 열 이름과 글자까지 같아야 합니다.
- 셀 끝의 제공 코드가 `rejected_rows`로 걸린 행의 데이터프레임 `rejected_table`을 만듭니다.

**예시**: 제17조 제3항(3개월, 30일)에 "받은 날부터 3개월 이내에 반품하실 수 있습니다."라고 쓴 답은 통과합니다. "받은 날부터 6개월 이내"라고 쓴 답은 6이 조문에 없어 `rejected_table`에 들어갑니다.

**확인 기준**
- `doc_texts`: `commerce_articles`의 문단마다 하나씩(39개) 있고 값이 정리한 본문입니다.
- `passed_rows`: `qa_rows` 가운데 두 검사를 모두 통과한 행이 `qa_rows` 순서대로 빠짐없이 들어 있습니다.
- `rejected_rows`, `rejected_table`: 딕셔너리의 키가 열 이름 다섯 개와 같아 테이블에 빈 칸(NaN)이 없고, 두 검사 칸이 `numbers_match`, `has_banned_phrase`의 결과와 같으며, 행마다 숫자 대조가 `False`이거나 금지 표현이 `True`입니다.
- 행 수: `passed_rows` 수와 `rejected_table` 행 수의 합이 `len(qa_rows)`입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 행마다 doc_id로 근거 조문을 찾아 두 검사를 한 번씩 하고, 결과에 따라 두 리스트 가운데 하나에 넣습니다.

세부구현:
1. doc_texts는 commerce_articles를 돌며 만드는 딕셔너리 컴프리헨션 한 줄입니다.
2. 두 검사 결과를 변수에 담아 두면 통과 여부 판단과 rejected_rows의 열에 함께 씁니다.
```

</details>

In [ ]:
# [작성] 답의 숫자와 금지 표현으로 Q&A 행을 검수하세요.

# (1) doc_id -> 정리한 본문
doc_texts = ...

# (2), (3) 통과한 행과 걸린 행 나누기
passed_rows = []
rejected_rows = []
for row in qa_rows:
    ...

# 걸린 행의 테이블(딕셔너리 키가 열 이름과 같아야 칸이 채워집니다)
rejected_table = pd.DataFrame(rejected_rows, columns=["id", "doc_id", "answer", "숫자 대조", "금지 표현"])
print("통과한 행 수:", len(passed_rows), "| 걸린 행 수:", len(rejected_table))
display(rejected_table.head(10))

In [ ]:
# [자가채점] 근거 조문 딕셔너리와 두 검사로 나눈 결과를 확인합니다.
assert isinstance(doc_texts, dict) and len(doc_texts) == len(commerce_articles), f"doc_texts가 {len(doc_texts)}개입니다. commerce_articles의 문단마다 하나씩({len(commerce_articles)}개), doc_id를 키로 만드세요. 생성 대상에서 뺀 문단도 7번 생성용 LLM이 근거로 고를 수 있어 넣어야 합니다."
for article in commerce_articles:
    assert doc_texts[article["doc_id"]] == article["text"], "doc_texts의 값은 1번에서 정리한 본문(text)입니다."

# 통과한 행: 숫자 대조 True, 금지 표현 False
for row in passed_rows:
    assert row in qa_rows, "passed_rows에는 qa_rows의 행 딕셔너리를 그대로 담습니다."
    assert numbers_match(row["answer"], doc_texts[row["doc_id"]]), f"{row['id']}는 숫자 대조에 걸리는데 passed_rows에 있습니다."
    assert not has_banned_phrase(row["answer"]), f"{row['id']}에는 금지 표현이 있는데 passed_rows에 있습니다."

# 두 검사를 통과해야 할 행이 빠짐없이 passed_rows에 들어갔는지 qa_rows 전체로 다시 확인합니다
expected_passed_ids = []
for row in qa_rows:
    if numbers_match(row["answer"], doc_texts[row["doc_id"]]) and not has_banned_phrase(row["answer"]):
        expected_passed_ids.append(row["id"])
passed_ids = []
for row in passed_rows:
    passed_ids.append(row["id"])
assert passed_ids == expected_passed_ids, f"passed_rows가 {len(passed_ids)}행인데 두 검사를 모두 통과한 행은 {len(expected_passed_ids)}행입니다. 통과한 행을 qa_rows 순서대로 빠짐없이 passed_rows에 담으세요."

# 걸린 행의 딕셔너리 키: 테이블의 열 이름과 같아야 빈 칸(NaN) 없이 테이블이 만들어집니다
assert isinstance(rejected_table, pd.DataFrame), "rejected_table은 데이터프레임입니다."
assert list(rejected_table.columns) == ["id", "doc_id", "answer", "숫자 대조", "금지 표현"], "rejected_table의 열은 id, doc_id, answer, 숫자 대조, 금지 표현입니다. 셀 끝의 제공 코드를 바꾸지 마세요."
for rejected in rejected_rows:
    assert list(rejected) == ["id", "doc_id", "answer", "숫자 대조", "금지 표현"], f"rejected_rows의 딕셔너리 키가 {list(rejected)}입니다. 키를 열 이름 id, doc_id, answer, 숫자 대조, 금지 표현과 글자까지 같게 쓰세요."

# 걸린 행: 두 검사 칸이 실제 검사 결과와 같고, 둘 중 하나 이상에 걸림
for index, rejected in rejected_table.iterrows():
    assert rejected["숫자 대조"] == numbers_match(rejected["answer"], doc_texts[rejected["doc_id"]]), f"{rejected['id']}의 숫자 대조 칸이 numbers_match 결과와 다릅니다. 검사 결과를 그대로 넣으세요."
    assert rejected["금지 표현"] == has_banned_phrase(rejected["answer"]), f"{rejected['id']}의 금지 표현 칸이 has_banned_phrase 결과와 다릅니다. 검사 결과를 그대로 넣으세요."
    assert (not rejected["숫자 대조"]) or rejected["금지 표현"], f"{rejected['id']}는 두 검사를 모두 통과했는데 rejected_table에 있습니다."
assert len(passed_rows) + len(rejected_table) == len(qa_rows), "passed_rows와 rejected_table의 행 수를 더하면 len(qa_rows)여야 합니다. 행마다 둘 중 한 곳에만 넣으세요."
print("✅ 통과!")

**결과 읽기**
- **걸린 행:** 모두 틀린 답은 아닙니다. "3개월"을 "90일"로 바꿔 쓴 맞는 답도 90이 조문에 없어 걸리므로 근거 조문과 나란히 읽습니다.
- **통과한 할루시네이션:** 단순 변심 반품 비용을 "판매자가 냅니다"로 바꾼 답처럼 숫자 없는 오류는 통과하므로, 사람이 표본을 읽어 확인합니다.
- **빠진 조건:** 제17조 제1항만 보고 만든 "7일 안이면 반품할 수 있습니다"에는 제2항의 제한 사유가 없습니다. 문단 하나 단위로 만드는 방법의 한계입니다.

## 6. 질문 늘리기

**배경**: 조문에서 만든 질문은 조문 문장을 따라가므로 온라인 쇼핑 이용자나 판매자가 자기 상황을 말하며 묻는 질문을 시드 지시 3개(셀의 `seed_instructions`)로 늘리고, 문서 생성 행이 적은 주제에는 모자란 만큼 더 요청해 주제 사이의 행 수 차이를 줄입니다.

**요구사항**
- (1) 새 지시 리스트를 담는 Pydantic 모델 **`InstructionSet`**(필드 `items`, 타입은 `list[str]`)을 만듭니다.
- (2) system 메시지 "온라인 쇼핑 이용자나 판매자가 상담 챗봇에 실제로 입력할 법한 질문을 써야 한다. 네가 쓴 질문은 챗봇 학습 데이터의 질문이 된다."와 셀에 적어 둔 `question_human`으로 프롬프트 템플릿 **`question_prompt`** 를 만듭니다.
- (3) `question_prompt`와 `InstructionSet` 모양으로만 답하는 `llm`을 이어 체인 **`instruction_generator`** 를 만듭니다.
- (4) 주제 태그 5개(셀의 `topics`, `source_counts`의 행 이름)마다 입력 딕셔너리 `{"topic": 태그, "seeds": seed_lines, "n": 개수}`(`seeds`는 셀의 `seed_lines`)를 차례로 **`instruction_inputs`** 에 담습니다.
  - 개수는 주제별 중앙값(`median_rows`)에서 그 주제의 문서 생성 행 수(셀의 `topic_row_counts`, 행이 없으면 0)를 뺀 값을 `int`로 바꾼 것입니다. 이 값이 `MIN_NEW_PER_TOPIC`(3)보다 작으면 3, `MAX_NEW_PER_TOPIC`(10)보다 크면 10으로 맞춥니다.
  - 셀 끝의 제공 코드가 주제, 개수, 문서 생성 행 수를 나란히 놓은 테이블 `instruction_plan`을 만듭니다.
- (5) `instruction_inputs`를 `instruction_generator.batch`에 `config={"max_concurrency": MAX_CONCURRENCY}`와 `return_exceptions=True`를 주어 한꺼번에 보낸 결과 리스트를 **`instruction_results`** 에 담습니다. 요청은 이 셀에서만 보냅니다.
  - 이 호출은 `get_usage_metadata_callback()`의 `with` 블록(`as usage`) 안에서 하고, 블록이 끝난 뒤 `usage.usage_metadata`를 `stage_usage`에 `"질문 늘리기"` 키로 남깁니다.
- (6) 결과를 차례로 보며 실패한 요청(예외 객체)은 건너뛰고 그 수를 정수 **`instruction_failed`** 에 셉니다.
- (7) 성공한 결과의 지시마다 `{"source": 요청한 주제 태그, "question": 지시}` 딕셔너리를 **`new_instructions`** 에 담습니다.
- (8) `new_instructions`를 차례로 보며, 비교 대상(시드 지시 + 이미 남긴 지시) 가운데 하나라도 `jaccard`가 0.8 이상이면 버리고 아니면 **`kept_instructions`** 에 담습니다. 남긴 지시는 다음 지시의 비교 대상에 더합니다.

**예시**
- 문서 생성 행이 주제별로 30, 25, 18, 12, 4행이면 중앙값은 18행이고, 모자란 행 수는 -12, -7, 0, 6, 14입니다. 하한 3과 상한 10으로 맞추면 `n`은 차례로 3, 3, 3, 6, 10입니다.
- 첫 시드 지시 "어제 받은 운동화가 작은데 단순 변심으로도 반품할 수 있나요?"와 끝맺음만 다른 "… 반품할 수 있을까요?"는 자카드 0.89라 버립니다. 같은 뜻의 "사이즈가 안 맞는 신발을 마음이 바뀐 이유로 돌려보내도 되나요?"는 0.14라 남습니다.

**확인 기준**
- `InstructionSet`, `instruction_generator`: 필드가 `items`이고, 체인이 빈자리 `topic`, `seeds`, `n`의 `question_prompt`로 시작해 `InstructionSet`을 돌려줍니다.
- `instruction_inputs`: `topics` 순서대로 주제마다 하나씩 `topic`, `seeds`, `n` 키의 딕셔너리이고, `n`은 위 규칙대로 모자란 행 수를 3~10으로 맞춘 값입니다.
- `instruction_results`, `instruction_failed`: 결과가 `instruction_inputs`와 같은 수이고 `stage_usage`에 `"질문 늘리기"` 키가 있으며, `instruction_failed`가 예외 객체 수와 같습니다.
- `new_instructions`: 성공한 결과의 지시 수를 모두 더한 만큼 있고, 딕셔너리가 `source`, `question` 두 키입니다. `topics` 순서대로 결과마다 그 결과를 요청한 주제 태그와 지시를 짝지어, 결과 안의 순서 그대로 담았습니다.
- `kept_instructions`: `new_instructions`의 딕셔너리를 순서대로 고른 리스트이고, 안의 어떤 두 질문도, 어떤 질문과 시드 지시도 자카드가 0.8 미만입니다.
- 버린 지시: 버린 시점의 비교 대상(시드 지시 + 앞에서 남긴 지시) 가운데 자카드 0.8 이상인 지시가 하나 이상 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 체인은 2번과 같은 모양이고, 주제별 개수와 거르기는 교안 01의 질문 늘리기와 같은 규칙입니다.

세부구현:
1. 개수는 반복문 안에서 if 두 개로 n을 하한과 상한 사이로 맞춘 뒤 딕셔너리를 더합니다. topic_row_counts는 Series라 get(주제, 0)으로 행이 없는 주제도 0으로 읽습니다.
2. 결과가 예외 객체인지 isinstance로 먼저 봅니다. 성공한 결과의 items가 지시 문자열 리스트입니다. 주제 태그 리스트와 결과를 zip으로 짝지어 지시마다 딕셔너리 하나를 더합니다.
3. 거를 때는 비교 대상을 하나씩 보다가 0.8 이상을 만나면 표시를 바꾸고 break로 멈춥니다. 표시가 그대로인 지시만 남기고 비교 대상에도 더합니다.
```

</details>

In [ ]:
# [작성] (1) 새 지시의 모양, (2) 프롬프트 템플릿, (3) 질문 늘리기 체인을 만드세요.


class InstructionSet(BaseModel):
    """새로 만든 지시 리스트입니다."""
    # items 필드(list[str], 새 지시 리스트)를 Field와 description으로 쓰세요
    ...


# 손으로 쓴 시드 지시 3개(단순 변심 반품, 배송 지연, 오픈마켓 책임)
seed_instructions = [
    "어제 받은 운동화가 작은데 단순 변심으로도 반품할 수 있나요?",
    "결제하고 일주일이 지났는데 아직 발송이 안 됐어요. 어떻게 해야 하나요?",
    "오픈마켓에서 산 물건이 불량인데 플랫폼에 책임을 물을 수 있나요?",
]

# human 메시지: {topic}(주제 태그), {seeds}(시드 지시 줄), {n}(만들 개수)
question_human = """온라인 쇼핑 이용자나 판매자가 상담 챗봇에 물어볼 '{topic}' 주제의 새 지시 {n}개를 만들어라. 지시는 챗봇에 입력하는 질문이고, 아래 [시드 지시]는 그 예시다.

질문 쓰는 법:
1. 시드 지시와 표현, 상황을 다르게 쓰고, 시드 지시를 그대로 옮기거나 낱말만 바꾸지 않는다.
2. 사실 확인, 절차, 문제 상황 질문을 섞는다.
3. 묻는 사람이 자기 상황을 말하며 묻는 말투로 쓴다.

[시드 지시]
{seeds}"""

# (2) 프롬프트 템플릿과 (3) 체인(InstructionSet 모양으로만 답함)
question_prompt = ...
instruction_generator = ...

In [ ]:
# [작성] (4) 주제마다 요청할 새 지시 개수를 정해 입력 딕셔너리를 만드세요.

# 시드 지시를 "- 지시" 줄로 잇습니다(모든 주제에 같은 시드)
seed_lines = "\n".join([f"- {seed}" for seed in seed_instructions])
topics = list(source_counts.index)

# 주제별 요청 개수: 중앙값(목표)에 모자란 만큼 요청하되 하한과 상한 사이로(과제는 문단이 39개라 교안보다 작게 둡니다)
MIN_NEW_PER_TOPIC = 3
MAX_NEW_PER_TOPIC = 10

# 주제별 문서 생성 행 수와 그 중앙값
topic_row_counts = qa_table["source"].value_counts()
median_rows = topic_row_counts.median()  # 주제마다 채울 목표 행 수

# 주제마다 입력 딕셔너리 만들기
instruction_inputs = []
for topic in topics:
    ...

# 주제별 문서 생성 행 수와 요청 개수를 확인합니다(seeds 칸은 모든 주제에 같은 긴 글이라 뺍니다)
instruction_plan = pd.DataFrame(instruction_inputs)[["topic", "n"]]
instruction_plan["문서 생성 행 수"] = [topic_row_counts.get(topic, 0) for topic in topics]
display(instruction_plan)

In [ ]:
# [작성] (5) 주제별 입력을 한꺼번에 보내고 토큰 사용량을 남기세요. 요청은 이 셀에서만 보냅니다.

# get_usage_metadata_callback()의 with 블록(as usage) 안에서 instruction_inputs를 한꺼번에 보내 instruction_results에 담기(실패한 요청은 결과 자리에 예외 객체로)
...

# with 블록이 끝난 뒤 usage.usage_metadata를 "질문 늘리기" 키로 stage_usage에 넣기
...
print("받은 결과 수:", len(instruction_results))

In [ ]:
# [작성] (6) 실패한 요청을 세고, (7) 지시마다 한 행으로 모으세요(이 셀을 다시 실행해도 요청은 다시 보내지 않습니다).

# 지시마다 요청한 주제 태그를 붙여 모으기(실패한 요청은 건너뛰고 셉니다)
new_instructions = []
instruction_failed = 0
for topic, result in zip(topics, instruction_results):
    ...

print("실패한 요청 수:", instruction_failed, "| 새 지시 수:", len(new_instructions))
display(pd.DataFrame(new_instructions).head(10))

In [ ]:
# [작성] (8) 시드 지시나 이미 남긴 지시와 비슷한 지시를 버리세요.

# 비교 대상: 시드 지시 + 지금까지 남긴 지시
compared_texts = list(seed_instructions)

kept_instructions = []
for instruction in new_instructions:
    ...

print("남은 지시 수:", len(kept_instructions), "/", len(new_instructions))
display(pd.DataFrame(kept_instructions))

In [ ]:
# [자가채점] 체인 모양, 주제별 요청 개수, 새 지시의 키, 남긴 지시의 자카드를 확인합니다.
assert list(InstructionSet.model_fields) == ["items"], "InstructionSet의 필드는 items 하나입니다. `...`를 채웠는지 확인하세요."
assert len(seed_instructions) == 3, "시드 지시 3개는 바꾸지 마세요."
assert isinstance(instruction_generator, RunnableSequence) and instruction_generator.first is question_prompt, "instruction_generator는 question_prompt로 시작하는 체인입니다."
assert set(question_prompt.input_variables) == {"topic", "seeds", "n"}, f"question_prompt의 빈자리는 topic, seeds, n입니다. 지금: {question_prompt.input_variables}"
assert instruction_generator.output_schema.model_json_schema()["title"] == "InstructionSet", "instruction_generator가 InstructionSet을 돌려주지 않습니다. llm에 InstructionSet으로 구조화 출력을 거세요."

# 주제별 요청 개수: 중앙값에 모자란 행 수를 3~10으로 맞춘 값
assert isinstance(instruction_inputs, list) and len(instruction_inputs) == len(topics), "instruction_inputs는 topics의 주제마다 하나씩입니다."
for topic, item in zip(topics, instruction_inputs):
    assert list(item) == ["topic", "seeds", "n"], "instruction_inputs의 딕셔너리는 topic, seeds, n 세 키입니다."
    assert item["topic"] == topic and item["seeds"] == seed_lines, "topics 순서대로 topic에 주제 태그, seeds에 seed_lines를 넣으세요."
    topic_rows = topic_row_counts.get(topic, 0)
    expected_n = int(median_rows - topic_rows)
    if expected_n < 3:
        expected_n = 3
    if expected_n > 10:
        expected_n = 10
    assert item["n"] == expected_n, f"{topic}은 문서 생성 행이 {topic_rows}행, 중앙값이 {median_rows}라 n은 {expected_n}입니다(모자란 행 수를 3~10으로 맞춤). 지금: {item['n']}"

# 보낸 요청: 주제마다 결과 하나, 실패한 요청 수는 예외 객체 수
assert isinstance(instruction_results, list) and len(instruction_results) == len(instruction_inputs), "instruction_results는 instruction_inputs마다 하나씩입니다. return_exceptions=True를 주었는지 확인하세요."
assert "질문 늘리기" in stage_usage, "stage_usage에 \"질문 늘리기\" 키가 없습니다. 요청 셀에서 with 블록이 끝난 뒤 usage.usage_metadata를 이 키로 넣으세요."
exception_count = 0
for result in instruction_results:
    if isinstance(result, Exception):
        exception_count += 1
assert instruction_failed == exception_count, f"instruction_failed({instruction_failed})가 예외 객체 수({exception_count})와 다릅니다. 실패한 요청마다 하나씩 세세요."

# 새 지시: 성공한 결과의 지시 수를 모두 더한 만큼, source와 question 두 키
expected_instruction_count = 0
for result in instruction_results:
    if not isinstance(result, Exception):
        expected_instruction_count += len(result.items)
assert len(new_instructions) > 0, "new_instructions가 비어 있습니다. 결과의 items를 돌며 딕셔너리를 더하세요."
assert len(new_instructions) == expected_instruction_count, f"new_instructions가 {len(new_instructions)}개인데 성공한 결과의 지시는 모두 {expected_instruction_count}개입니다. 결과마다 items의 지시를 하나도 빠뜨리지 않고 담으세요."
for instruction in new_instructions:
    assert list(instruction) == ["source", "question"], "new_instructions의 딕셔너리는 source, question 두 키입니다."

# 새 지시: 결과마다 요청한 주제 태그와 그 결과의 지시를 순서대로 짝지은 리스트
expected_new = []
for topic, result in zip(topics, instruction_results):
    if not isinstance(result, Exception):
        for item in result.items:
            expected_new.append({"source": topic, "question": item})
assert new_instructions == expected_new, "new_instructions가 결과와 짝이 맞지 않습니다. topics와 instruction_results를 zip으로 짝지어, 성공한 결과의 지시마다 그 결과를 요청한 주제 태그를 source에 넣으세요."

# 남긴 지시는 new_instructions를 순서대로 고른 리스트
assert len(kept_instructions) > 0, "kept_instructions가 비어 있습니다. 비슷하지 않은 지시를 남기세요."
position = 0
for instruction in new_instructions:
    if position < len(kept_instructions) and instruction == kept_instructions[position]:
        position += 1
assert position == len(kept_instructions), "kept_instructions는 new_instructions의 딕셔너리를 순서대로 골라 담습니다."

# 남긴 지시끼리, 남긴 지시와 시드 지시는 자카드 0.8 미만
for index in range(len(kept_instructions)):
    question = kept_instructions[index]["question"]
    for seed in seed_instructions:
        assert jaccard(question, seed) < 0.8, f"'{question}'는 시드 지시와 자카드 0.8 이상인데 남았습니다. 비교 대상에 시드 지시를 넣었는지 확인하세요."
    for earlier in kept_instructions[:index]:
        assert jaccard(question, earlier["question"]) < 0.8, f"'{question}'는 앞에서 남긴 지시와 자카드 0.8 이상인데 남았습니다. 남긴 지시를 비교 대상에 더했는지 확인하세요."

# 버린 지시는 버린 시점의 비교 대상(시드 지시 + 앞에서 남긴 지시)에 자카드 0.8 이상인 지시가 있어야 합니다
replay_compared = list(seed_instructions)
position = 0
for instruction in new_instructions:
    if position < len(kept_instructions) and instruction == kept_instructions[position]:
        replay_compared.append(instruction["question"])
        position += 1
    else:
        found_similar = False
        for other in replay_compared:
            if jaccard(instruction["question"], other) >= 0.8:
                found_similar = True
                break
        assert found_similar, f"'{instruction['question']}'는 시드 지시와 앞에서 남긴 지시 모두와 자카드 0.8 미만인데 버렸습니다. 0.8 이상인 지시만 버리고 나머지는 모두 남기세요."
print("✅ 통과!")

**결과 읽기**
- **요청 개수:** 중앙값 이상인 주제는 하한 3, 공급·배송처럼 행이 적은 주제는 모자란 만큼(최대 10)을 요청합니다.
- **새 지시 수:** `n`은 프롬프트로 요청할 뿐 스키마가 강제하지 않아 요청한 수와 다를 수 있습니다.
- **남은 지시:** 자카드는 글자가 거의 같은 지시만 버리므로, 낱말만 다른 같은 질문이 남았는지 테이블에서 읽습니다.

## 7. 새 질문에 근거 조문과 답 만들기

**배경**: 6번에서 늘린 새 질문에는 답과 근거 조문이 없습니다. 교안 01처럼 검색기(하이브리드 검색, 재정렬은 수업 시간 관계상 끔)로 질문마다 관련 있는 조문을 찾고, 생성용 LLM이 그 조문을 보고 근거 `doc_id`와 답을 만듭니다. 생성용 LLM이 고른 `doc_id`가 그 질문에서 검색한 조문이 아니면 근거 없는 행으로 나눕니다.

**요구사항**
- (1) Pydantic 모델 **`GroundedAnswer`**(문자열 필드 `doc_id`, `answer`)를 만듭니다.
- (2) system 메시지 "주어진 전자상거래법 조문에 적힌 내용으로만 상담 질문에 답해야 한다. 네가 쓴 답은 온라인 쇼핑 상담 챗봇이 배울 예시가 된다."와 셀에 적어 둔 `answer_human`으로 프롬프트 템플릿 **`answer_prompt`** 를 만듭니다.
- (3) `answer_prompt`와 `GroundedAnswer` 모양으로만 답하는 `llm`을 이어 체인 **`answer_generator`** 를 만듭니다.
- (4) 셀에 불러 둔 `build_retriever(articles, collection_name, candidate_k, use_rerank)`로 `commerce_articles` 전체(39문단)의 검색기 **`commerce_retriever`** 를 만듭니다. `collection_name`은 `"commerce_law"`이고, 문단이 적어 키워드 검색과 의미 검색이 후보를 3개씩 찾으며(`candidate_k=3`), 재정렬은 끕니다(`use_rerank=False`).
- (5) `kept_instructions`의 지시마다 `commerce_retriever.invoke(지시의 question)`로 조문을 찾아 RRF 순서의 앞 3개만 남기고(`[:3]`), 남긴 Document마다 `metadata["doc_id"]`를 모은 리스트를 차례로 **`retrieved_doc_ids`** 에 담습니다(지시 하나에 리스트 하나).
- (6) 같은 지시마다 남긴 Document 3개를 `f"{doc_id}: {page_content}"` 줄로 만들어 찾은 순서대로 `"\n"`으로 이은 문자열을 `"articles"`에, 지시의 질문을 `"question"`에 넣은 딕셔너리를 차례로 **`answer_inputs`** 에 담습니다.
- (7) `answer_inputs`를 `answer_generator.batch`로 보내고(`config={"max_concurrency": MAX_CONCURRENCY}`, `return_exceptions=True`), 결과를 **`answer_results`** 에 담습니다.
  - 이 호출은 `get_usage_metadata_callback()`의 `with` 블록(`as usage`) 안에서 하고, 블록이 끝난 뒤 `usage.usage_metadata`를 `stage_usage`에 `"새 질문에 답 만들기"` 키로 남깁니다.
- (8) 결과를 세 갈래로 나눕니다. 실패한 요청(예외 객체) 수를 정수 **`failed_count`** 에 담습니다.
- (9) 근거 `doc_id`가 그 지시의 `retrieved_doc_ids` 리스트에 없는 결과("없음" 포함)는 키가 `source`, `question`, `doc_id`, `answer` 순서인 딕셔너리로 **`no_source_rows`** 에 담습니다.
- (10) 나머지 결과는 **`answer_rows`** 에 담습니다. 키는 `qa_rows`와 같은 `id`, `doc_id`, `source`, `method`, `question`, `answer` 순서이고, `id`는 `"cnq-0001"`부터, `method`는 `"질문 늘리기"`입니다.
- (9), (10)의 `source`와 `question`은 결과와 짝지은 지시(`kept_instructions`의 딕셔너리)의 값입니다. `source`는 그 지시의 주제 태그이고, `doc_id`와 `answer`는 생성용 LLM 결과의 값입니다.

**예시**: "단순 변심으로 반품하면 택배비를 누가 내나요?"의 검색 결과에 "전자상거래법 제18조 제9항"이 있고 생성용 LLM이 이 `doc_id`와 답을 돌려주면 `answer_rows`로, "적립금은 언제 사라지나요?"처럼 조문에 없는 질문에 "없음"을 돌려주면 `no_source_rows`로 갑니다.

**확인 기준**
- `GroundedAnswer`, `answer_generator`: 필드가 `doc_id`, `answer`이고, 체인이 빈자리 `articles`, `question`의 `answer_prompt`로 시작해 `GroundedAnswer`를 돌려줍니다.
- `commerce_retriever`: `build_retriever(..., use_rerank=False)`가 돌려준 하이브리드 검색기 `EnsembleRetriever`이고(재정렬을 켜면 `ContextualCompressionRetriever`가 되어 통과하지 못합니다), 키워드 검색과 의미 검색의 후보 수가 둘 다 3입니다.
- `retrieved_doc_ids`, `answer_inputs`: 둘 다 `kept_instructions`와 같은 수입니다. 지시마다 `doc_id`가 서로 다른 3개이고 모두 `doc_texts`에 있으며, `articles`는 그 `doc_id` 순서대로 `doc_id: 정리한 본문` 줄을 이은 문자열, `question`은 지시의 질문입니다.
- 세 갈래: `answer_results`가 `kept_instructions`와 같은 수이고 `stage_usage`에 `"새 질문에 답 만들기"` 키가 있으며, `failed_count`, `answer_rows` 수, `no_source_rows` 수의 합도 같습니다.
- `answer_rows`: 키 여섯 개가 위 순서이고 `id`가 `cnq-0001`부터 차례대로이며, `doc_id`가 그 지시에서 검색한 `doc_id` 가운데 하나입니다.
- `no_source_rows`: 키 네 개가 위 순서이고 `doc_id`가 그 지시에서 검색한 `doc_id`에 없습니다.
- 짝: 두 리스트의 행마다 `question`, `source`가 짝지은 지시와 같고 `doc_id`, `answer`가 생성용 LLM의 결과와 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01의 새 질문에 답 만들기와 같은 순서입니다. 검색기를 만들고, 지시마다 조문을 찾아 doc_id 리스트와 입력을 만든 뒤 한꺼번에 보내고 결과를 나눕니다.

세부구현:
1. 지시마다 invoke로 받은 Document 리스트를 [:3]으로 자르고, 남긴 Document마다 metadata의 doc_id와 page_content로 줄을 만들어 줄 리스트와 doc_id 리스트에 더합니다.
2. 결과를 나눌 때는 kept_instructions, retrieved_doc_ids, answer_results 세 리스트를 zip으로 짝짓고, 실패(예외 객체)를 먼저 본 뒤 근거 doc_id가 그 지시의 doc_id 리스트에 있는지 봅니다.
```

</details>

In [ ]:
# [작성] (1) 생성용 LLM이 돌려줄 모양, (2) 프롬프트 템플릿, (3) 답 만들기 체인을 만드세요.


class GroundedAnswer(BaseModel):
    """생성용 LLM이 검색으로 찾은 조문을 보고 쓴 답입니다."""
    # doc_id, answer 두 문자열 필드를 Field와 description으로 쓰세요
    ...


# human 메시지: {articles}(검색으로 찾은 조문), {question}(지시)
answer_human = """아래 [질문]에 [조문]만 근거로 답하라.

근거와 답할 범위:
1. 답의 근거가 된 조문의 doc_id를 하나 적는다.
2. [조문]으로 질문의 핵심에 답할 수 있으면 조문에 적힌 내용만으로 조건을 빠뜨리지 않고 답한다. 질문 가운데 [조문]이 다루지 않는 부분은 답에서 빼고 언급하지 않는다('안내해 드릴 수 없습니다', '제공된 조문만으로는', '정해져 있지 않습니다', '판단할 수 없습니다' 같은 문장을 쓰지 않는다).
3. 자기 상황이 조문의 요건에 해당하는지 묻는 질문이면 조문의 요건을 밝히고 '이런 경우라면 ~에 해당할 수 있습니다'처럼 요건에 맞춰 답한다.
4. [조문]으로 질문의 핵심에 답할 수 없을 때만 doc_id에 "없음"을 적고, 답은 "안내해 드릴 수 없는 질문입니다."로 쓴다.

답 쓰는 법:
5. 답은 상담원이 직접 하는 존댓말로 한두 문장 쓰고, '조문에 따르면'·'조문상'·'이 조문'·'이 조항'·'해당 조항'·'해당 규정' 같은 말은 쓰지 않는다.
6. 답에서 조문을 밝힐 때는 '전자상거래법 제18조'처럼 법령 이름과 조 번호만 쓰고, 항·호 번호(제○항, 제○호)는 쓰지 않는다.

[조문]
{articles}

[질문]
{question}"""

# (2) 프롬프트 템플릿과 (3) 체인(GroundedAnswer 모양으로만 답함)
answer_prompt = ...
answer_generator = ...

In [ ]:
# [작성] (4) 검색기를 만들고, (5), (6) 지시마다 조문을 찾아 doc_id 리스트와 체인 입력을 만드세요.
import sys

# 단원 폴더의 검색기 파일(law_retriever.py)을 import 할 수 있게 경로에 더합니다(교안 01과 같음)
sys.path.append(str(material_dir))
from law_retriever import build_retriever

# (4) commerce_articles 전체의 검색기: collection_name은 "commerce_law", candidate_k=3, use_rerank=False
# 수업 시간 관계상 재정렬은 일부러 끕니다(use_rerank=True로 켜면 순서가 더 정확하지만 GPU 없이는 느립니다)
commerce_retriever = ...

# (5), (6) 지시마다 검색한 doc_id 리스트와 {"articles": 조문 줄, "question": 지시의 질문} 입력
retrieved_doc_ids = []
answer_inputs = []
for instruction in kept_instructions:
    ...

# 첫 요청의 질문, 검색한 doc_id, {articles}에 들어갈 조문 줄
print("첫 질문:", answer_inputs[0]["question"])
print("검색한 doc_id:", retrieved_doc_ids[0])
print(answer_inputs[0]["articles"])

In [ ]:
# [작성] (7) 남은 지시마다 검색한 조문과 함께 생성용 LLM에 보내세요.

# get_usage_metadata_callback()의 with 블록(as usage) 안에서 answer_inputs를 한꺼번에 보내 answer_results에 담기(실패한 요청은 예외 객체로)
...

# with 블록이 끝난 뒤 usage.usage_metadata를 "새 질문에 답 만들기" 키로 stage_usage에 넣기
...
print("보낸 요청 수:", len(answer_inputs))

In [ ]:
# [작성] 결과를 (8) 실패, (9) 근거 없음, (10) 근거 있음 세 갈래로 나누세요.
answer_rows = []
no_source_rows = []
failed_count = 0
for instruction, question_doc_ids, result in zip(kept_instructions, retrieved_doc_ids, answer_results):
    ...

print("실패한 요청 수:", failed_count, "| 근거 있는 행 수:", len(answer_rows), "| 근거 없는 행 수:", len(no_source_rows))
no_source_table = pd.DataFrame(no_source_rows, columns=["source", "question", "doc_id", "answer"])
display(no_source_table.head(10))

# 근거 있는 행: 답에 거절 문장이 섞였는지 읽어 봅니다
display(pd.DataFrame(answer_rows).head(10))

In [ ]:
# [자가채점] 답 만들기 체인, 검색기, 검색 결과와 입력, 세 갈래로 나눈 결과를 확인합니다.
assert list(GroundedAnswer.model_fields) == ["doc_id", "answer"], "GroundedAnswer의 필드는 doc_id, answer 두 개입니다. `...`를 채웠는지 확인하세요."
assert isinstance(answer_generator, RunnableSequence) and answer_generator.first is answer_prompt, "answer_generator는 answer_prompt로 시작하는 체인입니다."
assert set(answer_prompt.input_variables) == {"articles", "question"}, f"answer_prompt의 빈자리는 articles, question입니다. 지금: {answer_prompt.input_variables}"
assert answer_generator.output_schema.model_json_schema()["title"] == "GroundedAnswer", "answer_generator가 GroundedAnswer를 돌려주지 않습니다. llm에 GroundedAnswer로 구조화 출력을 거세요."

# 검색기: build_retriever(..., use_rerank=False)가 돌려준 하이브리드 검색기이고, 두 검색 모두 후보 3개씩
from langchain_classic.retrievers import EnsembleRetriever

assert isinstance(commerce_retriever, EnsembleRetriever), "commerce_retriever는 build_retriever(..., use_rerank=False)가 돌려준 하이브리드 검색기(EnsembleRetriever)입니다. `...`를 채웠는지, use_rerank=False를 줬는지 확인하세요."
assert commerce_retriever.retrievers[0].k == 3, "키워드 검색의 후보 수가 3이 아닙니다. build_retriever에 candidate_k=3을 주세요."
assert commerce_retriever.retrievers[1].search_kwargs["k"] == 3, "의미 검색의 후보 수가 3이 아닙니다. build_retriever에 candidate_k=3을 주세요."

# 검색 결과와 입력: 지시마다 하나씩, 검색한 doc_id 순서대로 "doc_id: 정리한 본문" 줄을 이은 articles
assert isinstance(retrieved_doc_ids, list) and len(retrieved_doc_ids) == len(kept_instructions), "retrieved_doc_ids는 kept_instructions의 지시마다 doc_id 리스트 하나씩입니다."
assert isinstance(answer_inputs, list) and len(answer_inputs) == len(kept_instructions), "answer_inputs는 kept_instructions의 지시마다 입력 딕셔너리 하나씩입니다."
for instruction, question_doc_ids, answer_input in zip(kept_instructions, retrieved_doc_ids, answer_inputs):
    assert isinstance(question_doc_ids, list) and len(question_doc_ids) == 3, f"'{instruction['question']}'의 검색한 doc_id 리스트는 invoke 결과를 [:3]으로 자른 3개입니다. 지금: {question_doc_ids}"
    assert len(set(question_doc_ids)) == 3, f"'{instruction['question']}'의 검색한 doc_id가 겹칩니다. 지시마다 검색을 한 번만 하고 그 결과의 앞 3개를 담으세요. 지금: {question_doc_ids}"
    for doc_id in question_doc_ids:
        assert doc_id in doc_texts, f"검색한 doc_id({doc_id})가 조문에 없습니다. Document의 metadata['doc_id']를 담으세요."
    assert set(answer_input) == {"articles", "question"}, "answer_inputs의 딕셔너리 키는 articles, question입니다."
    assert answer_input["question"] == instruction["question"], f"입력의 question이 지시 '{instruction['question']}'와 다릅니다."
    expected_lines = []
    for doc_id in question_doc_ids:
        expected_lines.append(f"{doc_id}: {doc_texts[doc_id]}")
    assert answer_input["articles"] == "\n".join(expected_lines), f"'{instruction['question']}'의 articles 모양이 다릅니다. 찾은 Document마다 'doc_id: page_content' 줄을 만들어 찾은 순서대로 줄바꿈으로 이으세요."

# 세 갈래의 합은 남긴 지시 수
assert len(answer_results) == len(kept_instructions), "answer_results는 kept_instructions마다 하나씩입니다."
assert "새 질문에 답 만들기" in stage_usage, "stage_usage에 \"새 질문에 답 만들기\" 키가 없습니다. 요청 셀에서 with 블록이 끝난 뒤 usage.usage_metadata를 이 키로 넣으세요."
assert failed_count + len(answer_rows) + len(no_source_rows) == len(kept_instructions), "실패, 근거 있음, 근거 없음 세 갈래의 수를 더하면 len(kept_instructions)여야 합니다. 결과마다 한 갈래에만 넣으세요."

# 근거 있는 행
for number, row in enumerate(answer_rows, start=1):
    assert list(row) == ["id", "doc_id", "source", "method", "question", "answer"], "answer_rows의 행 키는 qa_rows와 같은 id, doc_id, source, method, question, answer 순서입니다."
    assert row["id"] == f"cnq-{number:04d}", f"{number}번째 행의 id가 {row['id']}입니다. cnq-0001부터 차례대로 붙이세요."
    assert row["method"] == "질문 늘리기", "method는 \"질문 늘리기\"입니다."

# 근거 없는 행
for row in no_source_rows:
    assert list(row) == ["source", "question", "doc_id", "answer"], "no_source_rows의 딕셔너리 키는 source, question, doc_id, answer 순서입니다."

# 짝: 실패하지 않은 결과를 차례로 보며, 근거 doc_id가 그 지시에서 검색한 doc_id에 있으면 answer_rows, 없으면 no_source_rows의 다음 행과 맞는지 봅니다
answer_position = 0
no_source_position = 0
for instruction, question_doc_ids, result in zip(kept_instructions, retrieved_doc_ids, answer_results):
    if isinstance(result, Exception):
        continue
    if result.doc_id in question_doc_ids:
        assert answer_position < len(answer_rows), "근거 doc_id가 그 지시에서 검색한 조문인 결과보다 answer_rows가 적습니다. 그런 결과마다 한 행씩 넣으세요."
        row = answer_rows[answer_position]
        answer_position += 1
    else:
        assert no_source_position < len(no_source_rows), "근거 doc_id가 그 지시에서 검색한 조문에 없는 결과보다 no_source_rows가 적습니다. 그런 결과마다 한 행씩 넣으세요. 실제 조문이어도 그 지시에서 검색하지 않은 doc_id면 근거 없는 행입니다."
        row = no_source_rows[no_source_position]
        no_source_position += 1
    assert row["question"] == instruction["question"], f"'{row['question']}'가 짝지은 지시 '{instruction['question']}'와 다릅니다. kept_instructions, retrieved_doc_ids, answer_results를 zip으로 짝지어 그 지시의 question을 넣으세요."
    assert row["source"] == instruction["source"], f"'{row['question']}'의 source({row['source']})가 짝지은 지시의 주제 태그({instruction['source']})와 다릅니다. source에는 지시의 주제 태그를 넣으세요."
    assert row["doc_id"] == result.doc_id and row["answer"] == result.answer, f"'{row['question']}'의 doc_id나 answer가 생성용 LLM의 결과와 다릅니다. 결과의 doc_id와 answer를 그대로 넣으세요."
print("✅ 통과!")

**결과 읽기**
- **근거 없는 행:** 숫자 대조를 할 수 없어 학습 데이터에서 뺍니다. "없음"은 검색한 조문 3개로 질문의 핵심에 답할 수 없다는 뜻이라, 적립금처럼 법이 정하지 않은 질문인지 검색이 근거 조문을 놓친 질문인지 읽어 봅니다.
- **"없음"이 많을 때:** 근거 조문이 검색 결과에 없었으면 `candidate_k`와 남기는 수(`[:3]`)를 늘리거나 `use_rerank=True`로 재정렬을 켜고, 검색 결과에 있는데도 "없음"이면 `answer_human`의 규칙을 고칩니다.
- **근거 있는 행:** `doc_id`가 틀렸거나 숫자를 바꿨을 수 있고 거절 문장이 섞일 수 있어, 9번에서 같은 검수를 거친 행만 저장합니다.

## 8. 범위 밖 질문의 거절 행 만들기

**배경**: 근거 있는 답만 학습한 모델은 전자상거래법으로 답할 수 없는 질문에도 단정해서 답하므로, 모르는 질문에 답하지 않는 예시(거절 행)를 전체의 몇 퍼센트만 섞습니다.

**요구사항**
- (1) 새 체인을 만들지 않고 6번의 `instruction_generator`를 다시 씁니다. 셀에 적어 둔 범위 밖 주제 `OUT_OF_SCOPE_TOPIC`, 6번의 `seed_lines`, 개수 `OUT_OF_SCOPE_COUNT`(12)를 `{"topic": ..., "seeds": ..., "n": ...}`로 한 번 `invoke`한 결과를 **`out_of_scope_result`** 에 담습니다. 과제 02처럼 증강과 기존 FAQ를 더하면 범위 밖 행의 비율이 낮아지므로 다듬은 뒤 3~5% 안팎을 목표로 정한 개수이고, 실제 비율은 과제 02와 같은 방식으로 다듬은 뒤 확인합니다.
  - 이 호출은 `get_usage_metadata_callback()`의 `with` 블록(`as usage`) 안에서 하고, 블록이 끝난 뒤 `usage.usage_metadata`를 `stage_usage`에 `"범위 밖 질문 만들기"` 키로 남깁니다.
- (2) 받은 질문 리스트의 앞에서부터 `OUT_OF_SCOPE_COUNT`개(`out_of_scope_result.items[:OUT_OF_SCOPE_COUNT]`)마다 행 딕셔너리를 **`refusal_rows`** 에 담습니다. 스키마가 개수를 강제하지 않아 더 많이 올 수 있으므로 잘라 쓰고, 적게 오면 받은 만큼만 씁니다.
  - 키는 `id`, `doc_id`, `source`, `method`, `question`, `answer` 순서이고, `id`는 `"out-0001"`부터 네 자리 번호, `doc_id`는 `"없음"`, `source`와 `method`는 `"범위 밖"`, `answer`는 셀에 적어 둔 거절 답 `REFUSAL_ANSWER`입니다. 생성용 LLM이 붙인 유형이 없으므로 `question_type` 키는 넣지 않습니다.

**예시**: 받은 질문이 "해외 직구로 산 물건값도 연말정산 때 공제받을 수 있나요?"이면 첫 행은 `{"id": "out-0001", "doc_id": "없음", "source": "범위 밖", "method": "범위 밖", "question": "해외 직구로 …", "answer": "이 상담 챗봇이 다루는 전자상거래법 조문으로는 답할 수 없는 질문입니다. …"}`입니다.

**확인 기준**
- `out_of_scope_result`: `instruction_generator`가 돌려준 `InstructionSet`이고, `stage_usage`에 `"범위 밖 질문 만들기"` 키가 있습니다.
- `refusal_rows`: `out_of_scope_result.items[:OUT_OF_SCOPE_COUNT]`의 질문마다 한 행이 같은 순서로 있고(`OUT_OF_SCOPE_COUNT`개 이하), 행마다 키 여섯 개가 위 순서이며 `id`가 `out-0001`부터 차례대로입니다. `doc_id`, `source`, `method`, `answer`가 위 값입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01의 범위 밖 거절 행과 같은 방법입니다. 6번 체인에 주제만 바꿔 한 번 보내고, 돌려받은 질문마다 같은 거절 답을 붙입니다.

세부구현:
1. 입력 딕셔너리의 키는 6번 instruction_inputs와 같습니다. 주제 하나만 보내므로 .batch가 아니라 .invoke를 씁니다.
2. id 번호는 지금까지 모은 행 수에 1을 더한 값입니다.
```

</details>

In [ ]:
# [작성] 범위 밖 질문을 받아 (1) 결과를 담고, (2) 거절 답을 붙인 행으로 모으세요.

# 이 챗봇이 다루는 전자상거래법으로 답할 수 없는 주제, 그 질문에 붙일 거절 답, 요청 개수
OUT_OF_SCOPE_TOPIC = "이 챗봇이 다루는 전자상거래법으로는 답할 수 없는 질문(해외 직구 관세, 연말정산 같은 세금, 개인 투자, 이직 준비)"
REFUSAL_ANSWER = "이 상담 챗봇이 다루는 전자상거래법 조문으로는 답할 수 없는 질문입니다. 이 내용은 담당 기관이나 전문가에게 문의해 주세요."
OUT_OF_SCOPE_COUNT = 12  # 다듬은 뒤 범위 밖 비율 3~5% 안팎을 목표로 정한 개수

# (1) get_usage_metadata_callback()의 with 블록(as usage) 안에서 6번 체인을 .invoke로 한 번 불러 out_of_scope_result에 담기
...

# with 블록이 끝난 뒤 usage.usage_metadata를 "범위 밖 질문 만들기" 키로 stage_usage에 넣기
...

# (2) 질문마다 거절 답을 붙인 행(스키마가 개수를 강제하지 않아 더 많이 올 수 있으므로, 앞에서부터 OUT_OF_SCOPE_COUNT개만 씁니다)
refusal_rows = []
for question in out_of_scope_result.items[:OUT_OF_SCOPE_COUNT]:
    ...

# 사람이 질문을 모두 읽을 수 있게 거절 행 전체를 테이블로 봅니다
print("거절 행 수:", len(refusal_rows))
display(pd.DataFrame(refusal_rows))

In [ ]:
# [자가채점] 범위 밖 질문의 결과와 거절 행을 확인합니다.
assert isinstance(out_of_scope_result, InstructionSet), "out_of_scope_result는 instruction_generator가 돌려준 InstructionSet입니다. 주제 하나만 보내므로 .invoke로 보내세요."
assert "범위 밖 질문 만들기" in stage_usage, "stage_usage에 \"범위 밖 질문 만들기\" 키가 없습니다. with 블록이 끝난 뒤 usage.usage_metadata를 이 키로 넣으세요."
# 받은 질문 가운데 앞에서부터 OUT_OF_SCOPE_COUNT개만 씁니다(적게 오면 받은 만큼)
used_questions = out_of_scope_result.items[:OUT_OF_SCOPE_COUNT]
assert isinstance(refusal_rows, list) and len(refusal_rows) == len(used_questions), f"refusal_rows가 {len(refusal_rows)}행인데 받은 질문 {len(out_of_scope_result.items)}개 가운데 쓸 질문은 {len(used_questions)}개입니다. out_of_scope_result.items[:OUT_OF_SCOPE_COUNT]의 질문마다 한 행씩 담으세요."
assert len(refusal_rows) <= OUT_OF_SCOPE_COUNT, f"refusal_rows가 {len(refusal_rows)}행입니다. OUT_OF_SCOPE_COUNT({OUT_OF_SCOPE_COUNT})개를 넘지 않게 앞에서부터 잘라 쓰세요."
for index in range(len(refusal_rows)):
    row = refusal_rows[index]
    assert list(row) == ["id", "doc_id", "source", "method", "question", "answer"], "refusal_rows의 행 키는 id, doc_id, source, method, question, answer 순서입니다. question_type은 넣지 않습니다."
    assert row["id"] == f"out-{index + 1:04d}", f"{index + 1}번째 행의 id가 {row['id']}입니다. out-0001부터 네 자리 번호로 차례대로 붙이세요."
    assert row["doc_id"] == "없음", f"{row['id']}의 doc_id는 \"없음\"입니다."
    assert row["source"] == "범위 밖" and row["method"] == "범위 밖", f"{row['id']}의 source와 method는 \"범위 밖\"입니다."
    assert row["question"] == used_questions[index], f"{row['id']}의 question이 받은 질문과 다릅니다. 받은 순서대로 넣으세요."
    assert row["answer"] == REFUSAL_ANSWER, f"{row['id']}의 answer는 REFUSAL_ANSWER입니다."
print("✅ 통과!")

**결과 읽기**
- **사람이 읽을 것:** 반품 기한처럼 전자상거래법으로 답할 수 있는 질문이 섞였으면 자가채점을 통과한 뒤 그 행을 빼고 9번으로 넘어갑니다. 그런 질문에 거절 답을 학습하면 답해야 할 질문까지 거절합니다.
- **7번의 근거 없는 행을 쓰지 않는 이유:** 다른 주제의 조문으로 답할 수 있는 질문이 섞여 있습니다.
- **비율:** 증강과 기존 FAQ를 더하기 전이라 지금은 목표보다 높을 수 있고, 과제 02처럼 다듬은 뒤의 비율로 판단합니다.

## 9. 검수를 통과한 행과 거절 행을 JSONL로 저장하기

**배경**: 파인튜닝 데이터로 다듬는 다음 단계로 넘길 수 있게, 문서 생성 행과 질문 늘리기 행 가운데 검수를 통과한 행과 범위 밖 거절 행을 파일로 남깁니다.

**요구사항**
- (1) `passed_rows` 뒤에 `answer_rows` 가운데 `numbers_match`가 `True`이고 `has_banned_phrase`가 `False`인 행을 순서대로 이어 붙이고, 맨 끝에 `refusal_rows`를 더해 **`dataset_rows`** 에 담습니다. 거절 행은 근거 조문이 없어 숫자 대조 대상이 아닙니다.
- (2) `dataset_rows`를 한 줄에 JSON 하나씩(`json.dumps(행, ensure_ascii=False)`) **`dataset_path`**(`output/commerce_qa.jsonl`)에 저장합니다.

**확인 기준**
- `dataset_rows`: `passed_rows`로 시작해 `refusal_rows`로 끝나고, 행 수가 `passed_rows` 수, 두 검사를 통과한 `answer_rows` 수, `refusal_rows` 수의 합입니다.
- 파일: 줄 수가 `len(dataset_rows)`와 같고, 한글이 `\uXXXX`가 아닌 글자 그대로 저장돼 있습니다.
- 저장한 행: 범위 밖 행은 `doc_id`가 "없음"이고 답이 `REFUSAL_ANSWER`이며, 나머지 행은 모두 숫자 대조 `True`, 금지 표현 `False`입니다. `method`는 `"문서 생성"`, `"질문 늘리기"`, `"범위 밖"` 안에 있고 `"문서 생성"` 행이 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 문서 생성 행은 5번에서 이미 검수했으므로 그대로 두고, 질문 늘리기 행에만 같은 두 검사를 해 통과한 행을 더한 뒤 거절 행을 붙입니다.

세부구현:
1. 거절 행은 검수 반복문이 끝난 뒤 리스트 더하기로 붙입니다.
2. 파일은 쓰기 모드("w")와 utf-8 인코딩으로 열고, 행마다 JSON 문자열 끝에 줄바꿈을 붙여 씁니다.
```

</details>

In [ ]:
# [작성] 검수를 통과한 행과 거절 행을 모아 JSONL로 저장하세요.

# (1) 문서 생성 통과 행 + 두 검사를 통과한 질문 늘리기 행
dataset_rows = list(passed_rows)
for row in answer_rows:
    ...

# 거절 행은 근거 조문이 없어 숫자 대조 대상이 아니므로 검수 뒤에 더합니다
dataset_rows = ...

# (2) 한 줄에 행 하나씩 JSON으로 저장
dataset_path = output_dir / "commerce_qa.jsonl"
...

print("저장한 행 수:", len(dataset_rows), "| 파일:", dataset_path)
display(pd.DataFrame(dataset_rows)["method"].value_counts())

In [ ]:
# [자가채점] 저장한 파일의 줄 수, 한글 표기, 검수 결과, 방법을 확인합니다.
assert isinstance(dataset_rows, list) and dataset_rows[:len(passed_rows)] == passed_rows, "dataset_rows는 passed_rows로 시작하고, 그 뒤에 통과한 질문 늘리기 행과 거절 행을 붙입니다."
assert dataset_rows[len(dataset_rows) - len(refusal_rows):] == refusal_rows, "dataset_rows는 refusal_rows로 끝나야 합니다. 검수 반복문이 끝난 뒤 refusal_rows를 더하세요."

# 두 검사를 통과한 질문 늘리기 행 수와 거절 행 수만큼 passed_rows 뒤에 붙었는지 확인합니다
answer_passed_count = 0
for row in answer_rows:
    if numbers_match(row["answer"], doc_texts[row["doc_id"]]) and not has_banned_phrase(row["answer"]):
        answer_passed_count += 1
expected_count = len(passed_rows) + answer_passed_count + len(refusal_rows)
assert len(dataset_rows) == expected_count, f"dataset_rows가 {len(dataset_rows)}행입니다. passed_rows {len(passed_rows)}행, 두 검사를 통과한 질문 늘리기 행 {answer_passed_count}행, 거절 행 {len(refusal_rows)}행을 더하면 {expected_count}행입니다."
assert dataset_path.exists(), f"{dataset_path} 파일이 없습니다. 파일을 쓰는 코드를 채웠는지 확인하세요."

# 저장한 파일을 줄 단위로 다시 읽습니다
saved_text = dataset_path.read_text(encoding="utf-8")
saved_lines = saved_text.splitlines()
assert len(saved_lines) == len(dataset_rows), f"파일의 줄 수({len(saved_lines)})가 dataset_rows의 행 수({len(dataset_rows)})와 다릅니다. 행마다 줄바꿈을 붙여 한 줄씩 쓰세요."
assert "\\u" not in saved_text, "한글이 \\uXXXX로 저장됐습니다. json.dumps에 ensure_ascii=False를 주세요."

# 거절 행은 doc_id "없음"과 거절 답, 나머지 행은 두 검사 통과
saved_methods = set()
for line in saved_lines:
    saved = json.loads(line)
    if saved["method"] == "범위 밖":
        assert saved["doc_id"] == "없음" and saved["answer"] == REFUSAL_ANSWER, f"{saved['id']}: 범위 밖 행은 doc_id \"없음\"과 REFUSAL_ANSWER 그대로 저장합니다."
    else:
        assert numbers_match(saved["answer"], doc_texts[saved["doc_id"]]), f"{saved['id']}는 숫자 대조에 걸리는데 저장됐습니다."
        assert not has_banned_phrase(saved["answer"]), f"{saved['id']}에는 금지 표현이 있는데 저장됐습니다."
    saved_methods.add(saved["method"])
assert saved_methods <= {"문서 생성", "질문 늘리기", "범위 밖"} and "문서 생성" in saved_methods, f"저장한 행의 method가 {saved_methods}입니다. 문서 생성 행, 질문 늘리기 행, 범위 밖 행만 저장합니다."
assert json.loads(saved_lines[0]) == dataset_rows[0], "파일의 첫 줄이 dataset_rows의 첫 행과 다릅니다. dataset_rows 순서대로 쓰세요."
print("✅ 통과!")

**결과 읽기**
- **방법별 행 수:** 조문을 따라가는 문서 생성 질문과 자기 상황을 말하는 질문 늘리기 질문이 함께 있어야 질문 말투가 다양해집니다.
- **다음 단계:** 이 파일은 교안 02의 증강, 품질 필터, 오염 검사, 구성 균형, chat 포맷, 분할을 거쳐야 학습에 넣을 수 있고, 과제 02가 같은 방식으로 만든 고정 파일로 진행합니다.

## 10. 단계별 실제 비용 확인하기

**배경**: API를 부른 단계마다 `stage_usage`에 모은 토큰 사용량으로 실제 비용을 계산하고, 3번에서 어림한 조문으로 Q&A 생성의 예상 비용이 맞았는지 확인합니다.

**요구사항**
- (1) `stage_usage`의 단계마다 키가 `"단계"`, `"실제 비용(달러)"` 순서인 딕셔너리를 **`cost_rows`** 에 담습니다. 실제 비용은 `usage_cost(그 단계의 usage_metadata)`를 `round(값, 4)`로 반올림한 값입니다.
- 셀 끝의 제공 코드가 `cost_rows`로 데이터프레임 `cost_table`을 만들어 보여 주고, 실제 비용 합계와 조문으로 Q&A 생성의 예상 비용(3번 `estimated_cost`)과 실제 비용을 출력합니다.

**예시**: `stage_usage`에 단계가 다섯 개면 `cost_rows`도 다섯 행이고, 첫 행은 `{"단계": "5개 문단 시험", "실제 비용(달러)": 시험 비용}`입니다.

**확인 기준**
- `stage_usage`: `"5개 문단 시험"`, `"조문으로 Q&A 생성"`, `"질문 늘리기"`, `"새 질문에 답 만들기"`, `"범위 밖 질문 만들기"` 다섯 키가 있습니다.
- `cost_table`: 열이 `"단계"`, `"실제 비용(달러)"` 순서이고 `stage_usage`의 단계마다 한 행을 같은 순서로 담습니다.
- 비용 칸: 실제 비용이 모두 0보다 크고 `round(usage_cost(stage_usage[단계]), 4)`와 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01의 API 비용 확인과 같은 방법입니다. 비용 계산은 제공 함수 usage_cost에 맡깁니다.

세부구현:
1. stage_usage.items()로 단계 이름과 usage_metadata를 함께 받습니다.
2. 딕셔너리 하나를 만들어 cost_rows에 append합니다.
```

</details>

In [ ]:
# [작성] (1) 단계마다 실제 비용을 한 행으로 모으세요.
cost_rows = []
for stage, usage_metadata in stage_usage.items():
    # "단계"와 "실제 비용(달러)"(usage_cost를 round(값, 4)로) 딕셔너리를 cost_rows에 더하기
    ...

cost_table = pd.DataFrame(cost_rows)
display(cost_table)
print("실제 비용 합계(달러):", round(cost_table["실제 비용(달러)"].sum(), 4))

# 조문으로 Q&A 생성은 3번에서 어림한 예상 비용과 비교합니다
print("조문으로 Q&A 생성 예상(달러):", round(estimated_cost, 4), "| 실제(달러):", round(usage_cost(stage_usage["조문으로 Q&A 생성"]), 4))

In [ ]:
# [자가채점] 단계 키와 단계별 실제 비용을 확인합니다.
expected_stage_names = ["5개 문단 시험", "조문으로 Q&A 생성", "질문 늘리기", "새 질문에 답 만들기", "범위 밖 질문 만들기"]
missing_stages = []
for stage in expected_stage_names:
    if stage not in stage_usage:
        missing_stages.append(stage)
assert missing_stages == [], f"stage_usage에 {missing_stages} 키가 없습니다. 그 단계의 요청 셀에서 with 블록이 끝난 뒤 usage.usage_metadata를 이 키로 넣으세요. 3번 셀을 다시 실행했다면 stage_usage가 비워지므로 4, 6, 7, 8번의 요청 셀도 다시 실행합니다."
assert len(stage_usage) == 5, f"stage_usage의 키는 단계 이름 다섯 개입니다. 지금: {list(stage_usage)}"

assert isinstance(cost_table, pd.DataFrame), "cost_table은 데이터프레임입니다. 셀 끝의 제공 코드를 바꾸지 마세요."
assert list(cost_table.columns) == ["단계", "실제 비용(달러)"], f"cost_table의 열이 {list(cost_table.columns)}입니다. 딕셔너리 키를 단계, 실제 비용(달러) 순서로 쓰세요."
assert list(cost_table["단계"]) == list(stage_usage), "cost_table은 stage_usage의 단계마다 한 행을 같은 순서로 담습니다."
for index, row in cost_table.iterrows():
    stage = row["단계"]
    assert row["실제 비용(달러)"] > 0, f"{stage}의 실제 비용이 0입니다. 그 단계의 요청이 모두 실패했는지, with 블록 안에서 요청을 보냈는지 확인하세요."
    assert row["실제 비용(달러)"] == round(usage_cost(stage_usage[stage]), 4), f"{stage}의 실제 비용이 {row['실제 비용(달러)']}입니다. usage_cost에 그 단계의 usage_metadata를 넣고 round(값, 4)로 반올림하세요."
print("✅ 통과!")

**결과 읽기**
- **예상과 실제:** 요청마다 추론 토큰 길이가 다르고 캐시 입력은 싸게 계산되어, 예상과 실제가 달라집니다.
- **단계별 비용:** 새 질문에 답 만들기는 요청마다 검색한 조문 3개가 들어가, 문단 하나를 넣는 조문으로 Q&A 생성보다 요청당 입력 토큰 수가 많습니다. 검색기의 임베딩 호출은 `stage_usage`에 들어가지 않습니다.